# 1. CROSS-ATTENTION with the NEW encoders (CA) and CROSS-ATTENTION + GATE (CAG) — AeroSync-Mamba v30

**Background (v29).**
- **F0 loses to D.** F0 uses the new encoders with no cross-modal token interaction; D uses the old encoders with
  LACA's content-only cross-attention and the lag bias × 0. The mean window macro-F1 delta was −0.014 to −0.024,
  depending on how it was computed.
- **F1 matches D.** F1 (new encoders + a global scalar content gate) landed almost exactly where D did.
- **Why a scalar gate may be weak.** Mamba's pre-norm blocks make each token's input scale-invariant, so a scalar
  gate acts mostly through the residual stream. Cross-attention instead changes token **content**.

**Question.** Does real cross-attention with the **new** small encoders close the gap to D and beat plain
concatenation? Does a gate on top of cross-attention help further?

| model | encoders | fusion | status |
|---|---|---|---|
| A, I, D, F0, F1, F2 | (v29) | (v29) | **loaded from v29's saved results, NOT retrained** |
| **CA** | v29's NEW encoders (audio sigmoid gate 80 → MLP; IMU Conv1d k7 → CNN) | **D's own LACA module** (bidirectional Q/K/V softmax, 4 heads, lag bias × 0) → D's `[H_a ; SEP ; H_s]` → 2× Mamba | trained here |
| **CAG** | same as CA | CA + F1's exact softmax gate MLP applied **after** cross-attention and **before** Mamba | trained here |

**CA vs D isolates the encoder change.** CA subclasses v24's `AeroSyncMamba` and replaces only its two encoders,
so its `forward`, LACA, HTT, SEP token, pooling, auxiliary heads and loss are D's own code. CA − D is therefore
*exactly* "new encoders vs old encoders, both with cross-attention": the clean version of the comparison v29 could not
make.

**Content-only, by design.** HTT's time embedding is OFF and LACA's β = 0.0, so no timestamp or lag information
enters. A hook verifies this functionally: test predictions must be bit-identical when every timestamp is replaced by
zero. Temporal and lag-aware variants are closed for this project, so none are tested here.

**Protocol: identical to v29.**
- Same code: v29's parse, metadata, paired-window and fold cells, copied verbatim.
- Same windows: the resulting windows are asserted equal to v29's stored hash (`44c04ca18ec7a131`), and the per-fold
  z-score hashes to v29's.
- Same splits and loss: the same 95 missions, 5 day-out folds and seed-2026 inner validation; class-weighted CE +
  0.1 InfoNCE + 0.1 KL.
- Same seeds and backbone: seeds 42 and 123 only, and real `mamba_ssm` only.

**Determinism is already established (v29, not re-run).** mamba_ssm's CUDA backward is non-deterministic at the
gradient level: max |Δ grad| 2e-11 and max |Δ step loss| 5e-7. This had **zero** effect on validation macro-F1: max
|Δ| 0.00e+00, against a 0.016 noise floor. All other components were bit-identical.

**⚠️ Methodological fix: every paired comparison is computed TWO ways.**
- **Pooled-probability method:** per fold, average each window's probabilities over the seeds, then compute
  macro-F1. This is what v29's summary used.
- **Seed-averaged-F1 method:** per fold, compute each seed's own macro-F1, then average over the seeds.

These can disagree when a fold has few windows. Both are printed for **every** comparison. A difference in win count
or in the sign of the effect is flagged explicitly, and no single headline number is given without both.

## 2. Setup
v29's setup, with v30's own output paths: `crossattn_results.csv`, `v30_probs/` and `v30_checkpoints/`. v29's
files are **only read, never written**. They are searched for in this order:
1. `/kaggle/working`, if v29 ran in the same session;
2. any attached `/kaggle/input/**` folder, e.g. v29's Kaggle notebook output added as an input;
3. the public project repo, cloned with `--depth 1` (`version 29/Results/`, `version 29/Npz/`).

In [ ]:
# ==============================================================================
# 2a. ENVIRONMENT, PATHS, DATASET (v29's cell with v30 output paths) + project repo for v29's saved results
# ==============================================================================
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"    # cuBLAS reproducibility; only effective before CUDA initializes

import ast, copy, csv, gc, glob, hashlib, json, math, random, re, subprocess, sys, time, warnings
from collections import Counter, defaultdict
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

IS_KAGGLE = Path("/kaggle/working").exists()
IS_COLAB = (not IS_KAGGLE) and ("google.colab" in sys.modules or Path("/content").exists())
WORK = Path("/kaggle/working") if IS_KAGGLE else (Path("/content") if IS_COLAB else Path.cwd() / "v30_outputs")
BASE = WORK / "tii"
WORK.mkdir(parents=True, exist_ok=True)
BASE.mkdir(parents=True, exist_ok=True)
REPO_URL = "https://github.com/tiiuae/UAV-Realistic-Fault-Dataset.git"


def _is_dataset_root(p):
    return p.is_dir() and all((p / str(c)).is_dir() for c in range(5))


_att = []
for _pat in ("/kaggle/input/*/Dataset", "/kaggle/input/*/*/Dataset", "/kaggle/input/*/*/*/Dataset"):
    _att += [Path(p) for p in sorted(glob.glob(_pat)) if _is_dataset_root(Path(p))]
if _att:
    DATA_ROOT = _att[0]
else:
    _repo = BASE / "UAV-Realistic-Fault-Dataset"
    if not _is_dataset_root(_repo / "Dataset"):
        print(f"Cloning {REPO_URL} (--depth 1) ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(_repo)], check=True)
    DATA_ROOT = _repo / "Dataset"
assert _is_dataset_root(DATA_ROOT)

CACHE_DIR = WORK / "v29_cache"                       # v29's parse cache format (reused if present in this session)
PROBS_DIR = WORK / "v30_probs"
CKPT_DIR = WORK / "v30_checkpoints"
RESULTS_CSV = WORK / "crossattn_results.csv"
TABLE_CSV = WORK / "v30_results_table.csv"
for _d in (CACHE_DIR, PROBS_DIR, CKPT_DIR):
    _d.mkdir(parents=True, exist_ok=True)

PROJECT_REPO_URL = "https://github.com/Sainava/AeroSync-Mamba.git"
PROJECT_REPO = BASE / "AeroSync-Mamba"
if not (PROJECT_REPO / ".git").exists():
    _r = subprocess.run(["git", "clone", "--depth", "1", PROJECT_REPO_URL, str(PROJECT_REPO)], capture_output=True, text=True)
    print("project repo clone:", "OK" if _r.returncode == 0 else f"FAILED (non-fatal): {(_r.stderr or _r.stdout)[-200:]}")
print(f"DATA_ROOT {DATA_ROOT} | v30 results {RESULTS_CSV} | probs {PROBS_DIR} | checkpoints {CKPT_DIR} | repo {PROJECT_REPO}")

In [ ]:
# ==============================================================================
# 2b. REAL mamba-ssm — install if missing, import, raise on failure (no fallback)
# ==============================================================================
import importlib, importlib.util
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


def _pip(args):
    print("  pip install " + " ".join(args))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], capture_output=True, text=True)
    if r.returncode != 0:
        print((r.stderr or r.stdout)[-2000:])
    return r.returncode == 0


if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU: real mamba-ssm needs a GPU. Enable a GPU accelerator and re-run.")
if importlib.util.find_spec("mamba_ssm") is None:
    _ok = (_pip(["packaging", "ninja"]) and _pip(["causal-conv1d>=1.2.0", "--no-build-isolation"])
           and _pip(["mamba-ssm", "--no-build-isolation"]))
    importlib.invalidate_caches()
    if not _ok:
        raise RuntimeError("mamba-ssm installation FAILED (see pip output). Check internet + GPU, restart, re-run.")
try:
    import mamba_ssm
    from mamba_ssm import Mamba as MAMBA_CLASS
except Exception as exc:
    raise RuntimeError(f"`from mamba_ssm import Mamba` failed: {exc!r}. No fallback is used.") from exc
device = torch.device("cuda")
print(f"mamba_ssm {getattr(mamba_ssm, '__version__', '?')} ({MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__}) | torch "
      f"{torch.__version__} | CUDA {torch.version.cuda} | GPU {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# 2c. CONSTANTS — v29's values (the verbatim v29 cells below read them) + v30's run matrix and v29 references
# ==============================================================================
# --- v30 run matrix ---
SEEDS = [42, 123]                  # search-phase budget, as v29; NO further seeds in this notebook
V30_MODELS = ["CA", "CAG"]
V29_MODELS = ["A", "I", "D", "F0", "F1", "F2"]
ALL_MODELS = V29_MODELS + V30_MODELS
FUSED_MODELS = {"D", "F0", "F1", "F2", "CA", "CAG"}
MODELS = ["A", "I", "D", "F0", "F1", "F2"]   # v29's list, read by v29's verbatim model-building code
NEW_MODELS = ["F0", "F1", "F2"]
RUN_F2 = True

# --- missions / days (v29) ---
N_CLASSES = 5
EXCLUDE_RATIO = 0.5
EXPECTED_EXCLUDED = {"rosbag2_2021_12_29-09_10_31", "rosbag2_2021_12_31-10_12_31",
                     "rosbag2_2021_12_31-09_58_13", "rosbag2_2021_12_31-10_04_22"}
EXPECTED_N_MISSIONS = 95
EXPECTED_CLASS_COUNTS = [18, 19, 18, 20, 20]
DAYS = ["2021-12-22", "2021-12-31", "2022-01-04", "2022-01-05", "2022-01-10"]
DAY_MERGE = {"2021-12-29": "2021-12-31"}
EXPECTED_DAY_COUNTS = [10, 36, 17, 15, 17]
SMALL_FOLD_DAY = "2021-12-22"
STEADY = (0.10, 0.90)

# --- inner validation (v29: fallback_2026) ---
INNER_VAL_RULE = "fallback_2026"
INNER_VAL_FRACTION = 0.15
INNER_VAL_FALLBACK_SEED = 2026
INNER_VAL_SEED_V27 = 2027

# --- audio features / paired windows (v29) ---
N_SLOTS, DROP_SLOT, FRAME = 6, 5, 256
KEEP_SLOTS = [s for s in range(N_SLOTS) if s != DROP_SLOT]
BUFFER_LEN = N_SLOTS * FRAME
KEEP_BINS = list(range(1, 17))
N_AUDIO_FEAT = len(KEEP_SLOTS) * len(KEEP_BINS)
AUDIO_STD_EPS = 1e-6
WINDOW_SEC = 2.56
STEP_SEC = 1.28
TA_AUDIO = 64
TS_SENSOR = 128
MIN_SENSOR_SAMPLES_PER_WINDOW = 10
N_SENSOR_CHANNELS = 6
SENSOR_STD_EPS = 1e-6

# --- Condition D / shared model config (v29) ---
N_AUDIO_BINS = N_AUDIO_FEAT
D_MODEL = 256
TIME_EMBED_FREQUENCIES = 64
LACA_NUM_HEADS = 4
LACA_BETA = 1.0                    # only if the lag bias were ON; D, CA and CAG all use beta = 0.0
LAG_BIAS_HIDDEN = 32
LAG_BIAS_PARAMS = 97
LAG_BIAS_LN_EPS = 1e-5
HTT_TIME_EMBED_ENABLED = False
LACA_LAG_BIAS_ENABLED = False
MAMBA_NUM_BLOCKS = 2
MAMBA_D_STATE = 16
MAMBA_D_CONV = 2
MAMBA_EXPAND = 2
EXPECTED_MAMBA_PARAMS = 874_496
DROPOUT = 0.15
LAMBDA_ALIGN = 0.1
LAMBDA_CONS = 0.1
INFONCE_TEMPERATURE = 0.07
IMU_PRECONV_KERNEL = 7
IMU_PRECONV_CHANNELS = 6
GATE_HIDDEN = 64

# --- optimization (v29) ---
EPOCHS = 20
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_MIN = 1e-6
GRAD_CLIP_NORM = 1.0
NUM_WORKERS = 2

# --- analysis (v29) ---
BOOTSTRAP_RESAMPLES = 2000
BOOTSTRAP_SEED = 2028
NOISE_FLOOR_F1 = 0.016
GATE_STD_THRESHOLD = 0.05
N_HASH_SAMPLE = 25
ATTN_SAMPLE_WINDOWS = 64           # test windows per fold (evenly spaced) for the cross-attention diagnostics
NEAR_UNIFORM_ENTROPY = 0.99        # mean normalized row entropy above this = attention near-uniform

# --- v29 references (from the v29 run's printed output) ---
V29_PAIRED_SHA = "44c04ca18ec7a131"
V29_N_WINDOWS = 5238
V29_FOLD = {0: ("b5726f1a0d498224", (3932, 704, 602)), 1: ("720c42d948aed91e", (2700, 564, 1974)),
            2: ("dd723497b43897b8", (3680, 621, 937)), 3: ("5359d0c5720fd958", (3852, 596, 790)),
            4: ("a9b3c1d3fd502d2b", (3678, 625, 935))}
V29_SEARCH = {"results_csv": ["/kaggle/working/gated_fusion_results.csv", "/kaggle/input/**/gated_fusion_results.csv",
                              "{repo}/version 29/Results/gated_fusion_results.csv"],
              "table_csv": ["/kaggle/working/v29_results_table.csv", "/kaggle/input/**/v29_results_table.csv",
                            "{repo}/version 29/Results/v29_results_table.csv"],
              "npz_dirs": ["/kaggle/working/v29_probs", "/kaggle/input/**/v29_probs", "{repo}/version 29/Npz",
                           "{repo}/version 29/v29_probs"]}
print(f"v30: train {V30_MODELS} x {len(DAYS)} folds x seeds {SEEDS} = {len(V30_MODELS) * len(DAYS) * len(SEEDS)} runs; "
      f"reuse v29's {V29_MODELS} (not retrained)")

In [ ]:
# ==============================================================================
# 2d. SEEDING, DETERMINISM FLAGS, HOOK HELPER
# ==============================================================================
def seed_all(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def make_generator(seed):
    g = torch.Generator()
    g.manual_seed(seed)
    return g


def worker_init_fn(worker_id):
    s = torch.initial_seed() % 2 ** 32
    np.random.seed(s)
    random.seed(s)


torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True, warn_only=True)   # section 5 switches it off and on to test its effect
seed_all(42)

HOOKS = defaultdict(list)


def hook(name, ok, msg, stop=True):
    ok = bool(ok)
    HOOKS[name].append((ok, msg))
    print(f"[HOOK {name}] {'PASS' if ok else 'FAIL'} — {msg}")
    if not ok and stop:
        raise RuntimeError(f"HOOK {name} FAILED: {msg}")
    return ok


def loud(msg):
    print("🚨" * 50)
    for line in msg.split("\n"):
        print("🚨  " + line)
    print("🚨" * 50)


def sha16(*arrays):
    h = hashlib.sha1()
    for a in arrays:
        h.update(np.ascontiguousarray(a).tobytes())
    return h.hexdigest()[:16]

## 3. Data — v29's cells, verbatim, then an assertion against v29's stored window hash
v29 did not save its window arrays, only their content hash. So the windows are produced by **v29's own cells,
copied unchanged**, and the result must reproduce v29's recorded content sha1 (`44c04ca18ec7a131`), window count and
counts. No new window construction is introduced.

In [ ]:
# ==============================================================================
# 3a. PARSE (all 99 folders: the 4 exclusions are computed from metadata, not assumed) -> v29_cache/<mission>.npz
# ==============================================================================
def unwrap_message(obj):                                   # IMU notebook, unchanged
    if isinstance(obj, dict):
        for key in ("msg", "message", "data"):
            if key in obj and isinstance(obj[key], dict):
                return obj[key]
    return obj


def get_vec(d, candidates):                                # IMU notebook, unchanged
    for key in candidates:
        if key in d:
            v = d[key]
            if isinstance(v, (list, tuple)) and len(v) >= 3:
                return np.asarray(v[:3], dtype=np.float64)
        vals, ok = [], True
        for i in range(3):
            k = key.format(i=i)
            if k in d:
                vals.append(d[k])
            else:
                ok = False
                break
        if ok:
            return np.asarray(vals, dtype=np.float64)
    return None


def parse_imu_file(path):
    """IMU notebook's parser: rows [timestamp_us, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z] in FILE ORDER."""
    rows, bad = [], 0
    with open(path, "r") as f:
        for line in f:
            try:
                obj = unwrap_message(json.loads(line))
                ts = obj.get("timestamp", obj.get("time_stamp", None))
                acc = get_vec(obj, ["accelerometer_m_s2", "accelerometer_m_s2[{i}]", "accelerometer", "acceleration"])
                gyro = get_vec(obj, ["gyro_rad", "gyro_rad[{i}]", "gyro", "angular_velocity"])
                if ts is None or acc is None or gyro is None:
                    bad += 1
                    continue
                rows.append([float(ts), *acc.tolist(), *gyro.tolist()])
            except Exception:
                bad += 1
    return np.asarray(rows, dtype=np.float64).reshape(-1, 7), bad


def parse_mission(mdir):
    cache = CACHE_DIR / f"{mdir.name}.npz"
    if cache.exists():
        z = np.load(cache)
        return {k: z[k] for k in z.files}
    s_path = sorted(mdir.glob("*SensorCombined*.jsonl"))[0]
    a_path = sorted(mdir.glob("*AudioBuffer*.jsonl"))[0]
    imu, bad = parse_imu_file(s_path)
    chunks, lens, rst, rsts = [], Counter(), None, set()
    with open(a_path, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            x = rec.get("data")
            if x is None or len(x) == 0:
                continue
            if "recording_started_time" in rec:
                rsts.add(float(rec["recording_started_time"]))
                if rst is None:
                    rst = float(rec["recording_started_time"])          # v24: the FIRST record's value
            lens[len(x)] += 1
            chunks.append(x)
    modal = lens.most_common(1)[0][0]
    assert modal == BUFFER_LEN, f"{mdir.name}: modal buffer length {modal}"
    assert rst is not None, f"{mdir.name}: no recording_started_time — the audio cannot be anchored"
    keep_idx = np.array([i for i, x in enumerate(chunks) if len(x) == modal], dtype=np.int64)
    raw = np.asarray([chunks[i] for i in keep_idx], dtype=np.int64)
    n_all = len(chunks)
    del chunks
    frames = raw.reshape(len(raw), FRAME, N_SLOTS)                # sample i -> frame i // 6, slot i % 6
    slot6_zero = bool(np.all(frames[:, :, DROP_SLOT] == 0))
    ch = frames[:, :, KEEP_SLOTS].astype(np.float64)
    ch = ch - ch.mean(axis=1, keepdims=True)                      # per-chunk mean removed, per channel
    pw = (np.abs(np.fft.rfft(ch, axis=1)) ** 2).transpose(0, 2, 1).astype(np.float32)    # (n, 5, 129)
    feats = (pw[:, :, KEEP_BINS] ** 0.25).reshape(len(pw), N_AUDIO_FEAT).astype(np.float32)   # sqrt|rFFT|, channel-major
    out = {"imu_t": imu[:, 0], "imu_x": imu[:, 1:].astype(np.float32), "imu_bad": bad, "feats": feats,
           "keep_idx": keep_idx, "n_all_chunks": n_all, "rst": rst, "n_rst_values": len(rsts), "slot6_zero": slot6_zero}
    tmp = CACHE_DIR / f"{mdir.name}.tmp.npz"
    np.savez(tmp, **out)
    os.replace(tmp, cache)
    return out


def mission_day(name):
    d = name[8:18].replace("_", "-")
    return DAY_MERGE.get(d, d)


ALL_DIRS, LABEL_OF = [], {}
for class_dir in sorted(DATA_ROOT.iterdir()):
    if not (class_dir.is_dir() and class_dir.name.isdigit()):
        continue
    for mdir in sorted(class_dir.iterdir()):
        if mdir.is_dir() and not mdir.name.startswith("."):
            ALL_DIRS.append(mdir)
            LABEL_OF[mdir.name] = int(class_dir.name)
_t0 = time.time()
PARSED = {}
for i, mdir in enumerate(ALL_DIRS, 1):
    PARSED[mdir.name] = parse_mission(mdir)
    if i % 10 == 0 or i == len(ALL_DIRS):
        print(f"  parsed {i}/{len(ALL_DIRS)} missions | {time.time() - _t0:.0f} s")

In [ ]:
# ==============================================================================
# 3b. METADATA — v24's corrected audio anchor, exclusion rule, mission / class / day counts (HOOK data)
# ==============================================================================
def timebase(P):
    """v24 CORRECTED_ALIGNMENT time axes. IMU: FILE ORDER, seconds from the first sample. Audio: anchored at
    recording_started_time; chunk duration = (mission duration - offset) / number of chunks."""
    t_us = P["imu_t"]
    s_times = (t_us - t_us[0]) / 1e6
    duration = float(s_times[-1] - s_times[0])
    offset = (float(P["rst"]) - float(t_us[0])) / 1e6
    chunk = (duration - offset) / int(P["n_all_chunks"])
    a_times = (offset + np.arange(int(P["n_all_chunks"]), dtype=np.float64) * chunk)[P["keep_idx"]]
    return s_times, a_times, duration, offset, chunk


META = []
for name, P in PARSED.items():
    s_times, a_times, dur, off, ch = timebase(P)
    META.append({"mission": name, "class": LABEL_OF[name], "day": mission_day(name), "duration_s": dur, "offset_s": off,
                 "ratio": off / dur, "chunk_ms": 1e3 * ch, "n_imu": len(s_times), "imu_inversions": int((np.diff(s_times) < 0).sum()),
                 "n_chunks": int(P["n_all_chunks"]), "chunks_dropped": int(P["n_all_chunks"]) - len(P["keep_idx"]),
                 "rst_values": int(P["n_rst_values"]), "slot6_zero": bool(P["slot6_zero"])})
META = pd.DataFrame(META).set_index("mission")
EXCLUDED = set(META.index[META.ratio >= EXCLUDE_RATIO])
KEPT = sorted(set(META.index) - EXCLUDED)
M = META.loc[KEPT]
LABEL = {n: int(M.loc[n, "class"]) for n in KEPT}
DAY = {n: M.loc[n, "day"] for n in KEPT}
_cls = np.bincount(M["class"], minlength=N_CLASSES).tolist()
_days = [int((M.day == d).sum()) for d in DAYS]
print(META.loc[sorted(EXCLUDED), ["class", "day", "offset_s", "duration_s", "ratio"]].round(3).to_string())
print(f"\nKept {len(KEPT)} missions | max kept ratio {M.ratio.max():.3f} | median offset {M.offset_s.median():.2f} s | "
      f"chunk duration {M.chunk_ms.median():.2f} ms (median; v24's corrected-anchor formula) | "
      f"non-modal chunks dropped {int(M.chunks_dropped.sum())} | missions with IMU timestamp inversions "
      f"{int((M.imu_inversions > 0).sum())} (total {int(M.imu_inversions.sum())})")
print(pd.crosstab(M.day, M["class"]).reindex(DAYS).to_string())
hook("data", EXCLUDED == EXPECTED_EXCLUDED, f"exclusion set == the 4 known names: {sorted(EXCLUDED)}")
hook("data", len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and _days == EXPECTED_DAY_COUNTS,
     f"{len(KEPT)} missions, classes {_cls}, days {_days} (expected {EXPECTED_N_MISSIONS}, {EXPECTED_CLASS_COUNTS}, {EXPECTED_DAY_COUNTS})")
hook("data", bool(M.slot6_zero.all()) and bool((M.rst_values == 1).all()),
     f"audio slot 6 identically zero in {int(M.slot6_zero.sum())}/{len(M)}; recording_started_time constant in "
     f"{int((M.rst_values == 1).sum())}/{len(M)}")

In [ ]:
# ==============================================================================
# 3c. ONE SET OF PAIRED WINDOWS (v24 CORRECTED_ALIGNMENT): audio (64 x 80) and IMU (128 x 6, file order) from the SAME
#     real time span of the SAME mission. Every model below reads these arrays (unimodal models ignore one half).
# ==============================================================================
def paired_windows(name, P):
    s_times, a_times, dur, _, _ = timebase(P)
    tel, feats = P["imu_x"], P["feats"]
    max_time = min(s_times[-1], a_times[-1])
    out, i = [], 0
    t_start = max(0.0, float(a_times[0]))                  # v24: CORRECTED_WINDOW_START = "audio_onset"
    while t_start + WINDOW_SEC <= max_time:
        t_end = t_start + WINDOW_SEC
        s_mask = (s_times >= t_start) & (s_times <= t_end)
        if s_mask.sum() < MIN_SENSOR_SAMPLES_PER_WINDOW:
            t_start += STEP_SEC
            continue
        s_idx = np.where(s_mask)[0]
        pick = s_idx[np.linspace(0, len(s_idx) - 1, TS_SENSOR).round().astype(int)]
        tau_s_abs = s_times[pick]
        x_s = tel[pick].astype(np.float32)                 # real file-order samples (IMU notebook rule)
        x_s_v24 = np.stack([np.interp(tau_s_abs, s_times[s_mask], tel[s_mask, c]) for c in range(N_SENSOR_CHANNELS)],
                           axis=1).astype(np.float32)      # v24's np.interp form, kept only for the comparison count
        a_mask = (a_times >= t_start) & (a_times <= t_end)
        if a_mask.sum() >= 2:
            a_idx = np.where(a_mask)[0]
            a_pick = a_idx[np.linspace(0, len(a_idx) - 1, TA_AUDIO).round().astype(int)]
            tau_a_abs = a_times[a_pick]
            x_a = feats[a_pick].astype(np.float32)         # == v24's np.interp at exact (strictly increasing) chunk times
        else:
            tau_a_abs = np.linspace(t_start, t_end, TA_AUDIO)
            x_a = np.stack([np.interp(tau_a_abs, a_times, feats[:, c], left=0.0, right=0.0) for c in range(N_AUDIO_FEAT)],
                           axis=1).astype(np.float32)
        rel = (t_start + WINDOW_SEC / 2) / dur
        out.append({"window_id": f"{name}#{i:04d}", "mission": name, "cls": LABEL[name], "day": DAY[name],
                    "t_start": t_start, "t_end": t_end,
                    "phase": "start" if rel < STEADY[0] else ("end" if rel > STEADY[1] else "steady"),
                    "x_a": x_a, "x_s": x_s, "tau_a": ((tau_a_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "tau_s": ((tau_s_abs - t_start) / WINDOW_SEC).astype(np.float32),
                    "v24_interp_differs": bool(not np.array_equal(x_s, x_s_v24)),
                    "v24_interp_maxdiff": float(np.abs(x_s - x_s_v24).max())})
        i += 1
        t_start += STEP_SEC                                 # v24: accumulated float step
    return out


_W = []
for name in KEPT:
    _W.extend(paired_windows(name, PARSED[name]))
WMETA = pd.DataFrame([{k: v for k, v in w.items() if k not in ("x_a", "x_s", "tau_a", "tau_s")} for w in _W])
XA = np.stack([w["x_a"] for w in _W])                      # (N, 64, 80)
XS = np.stack([w["x_s"] for w in _W])                      # (N, 128, 6)
TAU_A = np.stack([w["tau_a"] for w in _W])                 # (N, 64)   in window units (v24 dataset: tau / window_sec)
TAU_S = np.stack([w["tau_s"] for w in _W])                 # (N, 128)
YW = WMETA.cls.values.astype(np.int64)
del _W
gc.collect()
print(f"Paired windows: {len(WMETA)} | XA {XA.shape} | XS {XS.shape} | per class {np.bincount(YW, minlength=N_CLASSES).tolist()} | "
      f"per phase {WMETA.phase.value_counts().to_dict()}")
print(f"IMU: windows where the file-order direct pick differs from v24's np.interp form: "
      f"{int(WMETA.v24_interp_differs.sum())} / {len(WMETA)} (max abs diff {WMETA.v24_interp_maxdiff.max():.4g}) — "
      "nonzero only where a window's IMU timestamps go backwards")

# ---- pairing hooks ----
_span_ok = bool((TAU_A >= -1e-6).all() and (TAU_A <= 1 + 1e-6).all() and (TAU_S >= -1e-6).all() and (TAU_S <= 1 + 1e-6).all())
hook("pairing", _span_ok and WMETA.window_id.is_unique,
     "every window's audio tokens AND IMU samples lie inside its own [t_start, t_start + 2.56 s] span of ONE mission "
     "(both come from a single record: same mission id, same span)")
_rng_h = np.random.default_rng(0)
_sample = np.sort(_rng_h.choice(len(WMETA), size=min(N_HASH_SAMPLE, len(WMETA)), replace=False))
_hash_ok = True
for j in _sample:
    r = WMETA.iloc[j]
    rebuilt = {w["window_id"]: w for w in paired_windows(r.mission, PARSED[r.mission])}[r.window_id]
    same = sha16(rebuilt["x_a"], rebuilt["x_s"]) == sha16(XA[j], XS[j]) and abs(rebuilt["t_start"] - r.t_start) < 1e-12
    _hash_ok &= same
hook("pairing", _hash_ok, f"hash check on {len(_sample)} sampled windows: stored (audio, IMU) == rebuilt from the raw "
                          f"mission at the recorded (mission, t_start)")
_fin = bool(np.isfinite(XA).all() and np.isfinite(XS).all())
hook("pairing", _fin and XA.shape[1:] == (TA_AUDIO, N_AUDIO_FEAT) and XS.shape[1:] == (TS_SENSOR, N_SENSOR_CHANNELS),
     f"shapes audio {XA.shape[1:]}, IMU {XS.shape[1:]}; all finite: {_fin}")
PAIRED_SHA = sha16(XA, XS)
print(f"Paired-window content sha1: {PAIRED_SHA}")

In [ ]:
# ==============================================================================
# 3d. v30 CHECK — the paired windows are IDENTICAL to v29's (stored hash, not a fresh definition)
# ==============================================================================
hook("v29-identity", PAIRED_SHA == V29_PAIRED_SHA and len(WMETA) == V29_N_WINDOWS,
     f"paired-window content sha1 {PAIRED_SHA} == v29's {V29_PAIRED_SHA}; {len(WMETA)} windows == v29's {V29_N_WINDOWS}")
hook("v29-identity", len(KEPT) == EXPECTED_N_MISSIONS and _cls == EXPECTED_CLASS_COUNTS and _days == EXPECTED_DAY_COUNTS,
     f"{len(KEPT)} missions, classes {_cls}, days {_days} — identical to v29")

## 4. Folds — v29's cell, verbatim, plus an assertion against v29's per-fold hashes

In [ ]:
# ==============================================================================
# 4. FOLDS (identical for all 6 models)
# ==============================================================================
def inner_val_fallback_2026(pool, k):
    """Standalone audio/IMU notebooks' rule: per class, ~15% (min 1) of the training missions -> validation, seed 2026."""
    rng = np.random.default_rng([INNER_VAL_FALLBACK_SEED, k])
    train, val = [], []
    for c in range(N_CLASSES):
        ms = sorted(m for m in pool if LABEL[m] == c)
        perm = rng.permutation(len(ms))
        n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
        val += [ms[i] for i in perm[:n_val]]
        train += [ms[i] for i in perm[n_val:]]
    return train, val


def inner_val_v27(pool, k):
    """v27/v28 `_split_inner_val(pool, protocol_idx=0, fold_idx=k)` (fingerprint 58e3c037c513ace2), seed 2027."""
    rng = np.random.default_rng([INNER_VAL_SEED_V27, 0, k])
    train, val = [], []
    for c in range(N_CLASSES):
        ms = sorted(m for m in pool if LABEL[m] == c)
        perm = rng.permutation(len(ms))
        n_val = max(1, int(round(INNER_VAL_FRACTION * len(ms))))
        assert len(ms) - n_val >= 1
        val += [ms[i] for i in perm[:n_val]]
        train += [ms[i] for i in perm[n_val:]]
    return train, val


inner_val = inner_val_fallback_2026 if INNER_VAL_RULE == "fallback_2026" else inner_val_v27
FOLDS = {}
for k, day in enumerate(DAYS):
    test = sorted(m for m in KEPT if DAY[m] == day)
    tr, va = inner_val([m for m in KEPT if DAY[m] != day], k)
    FOLDS[k] = {"day": day, "train": sorted(tr), "val": sorted(va), "test": test}
    assert not (set(tr) & set(va)) and not ((set(tr) | set(va)) & set(test)) and set(tr) | set(va) | set(test) == set(KEPT)
    assert day not in {DAY[m] for m in tr + va}
hook("folds", sorted(m for f in FOLDS.values() for m in f["test"]) == sorted(KEPT),
     "5 day-out test folds partition the 95 missions; no mission in two of train/val/test; test day never in training")

FOLD_PREP = {}
for k, f in FOLDS.items():
    idx = {s: np.where(WMETA.mission.isin(f[s]).values)[0] for s in ("train", "val", "test")}
    assert set(WMETA.mission.values[idx["train"]]) == set(f["train"])
    a_tr = XA[idx["train"]].reshape(-1, N_AUDIO_FEAT)
    s_tr = XS[idx["train"]].reshape(-1, N_SENSOR_CHANNELS)
    a_mu, a_sd = a_tr.mean(0, dtype=np.float64).astype(np.float32), (a_tr.std(0, dtype=np.float64) + AUDIO_STD_EPS).astype(np.float32)
    s_mu, s_sd = s_tr.mean(0, dtype=np.float64).astype(np.float32), (s_tr.std(0, dtype=np.float64) + SENSOR_STD_EPS).astype(np.float32)
    del a_tr, s_tr
    counts = np.bincount(YW[idx["train"]], minlength=N_CLASSES)
    FOLD_PREP[k] = {"idx": idx, "a_mu": a_mu, "a_sd": a_sd, "s_mu": s_mu, "s_sd": s_sd, "zhash": sha16(a_mu, a_sd, s_mu, s_sd),
                    "class_weights": (counts.sum() / (N_CLASSES * np.maximum(counts, 1))).astype(np.float32)}

print(f"Inner validation: {INNER_VAL_RULE}")
print(f"{'fold':<5}{'test day':<12}{'missions tr/val/test':<22}{'test missions/class':<22}{'windows tr/val/test':<22}{'z-score hash':<18}class weights")
for k, f in FOLDS.items():
    P = FOLD_PREP[k]
    print(f"{k:<5}{f['day']:<12}{'/'.join(str(len(f[s])) for s in ('train', 'val', 'test')):<22}"
          f"{str(np.bincount([LABEL[m] for m in f['test']], minlength=N_CLASSES).tolist()):<22}"
          f"{'/'.join(str(len(P['idx'][s])) for s in ('train', 'val', 'test')):<22}{P['zhash']:<18}"
          f"{np.round(P['class_weights'], 3).tolist()}" + ("  <- SMALL fold (10 missions)" if f["day"] == SMALL_FOLD_DAY else ""))

In [ ]:
# ==============================================================================
# 4b. v30 CHECK — z-score statistics and window counts per fold identical to v29's run
# ==============================================================================
_fold_ok = all(FOLD_PREP[k]["zhash"] == V29_FOLD[k][0] and
               tuple(len(FOLD_PREP[k]["idx"][s]) for s in ("train", "val", "test")) == V29_FOLD[k][1] for k in range(len(DAYS)))
hook("v29-identity", _fold_ok, "every fold's z-score hash and train/val/test window counts == v29's: " + ", ".join(
    f"fold {k} {FOLD_PREP[k]['zhash']}" for k in range(len(DAYS))))

## 5. Models
- **5a:** Condition D's classes, verbatim. This is v29's cell, which itself holds v24's source verbatim.
- **5b:** v29's A, I, new encoders and F0/F1/F2, verbatim. They are needed to reuse the encoders and F1's gate MLP,
  and for the structural checks. None of these models is trained here.
- **5c:** the new models.
  - **CA** = `class CrossAttnMamba(AeroSyncMamba)`. It inherits D's constructor and `forward` unchanged and replaces
    only `audio_encoder` (v29's `GatedAudioEncoder`) and `sensor_encoder` (v29's `ConvIMUEncoder`). D's `forward`
    passes the IMU to the encoder as (B, 6, T), so a two-line subclass of `ConvIMUEncoder` accepts that layout. It
    has the same layers, parameters and computation.
  - **CAG** = CA + F1's gate: the same `Linear(512→64) → GELU → Linear(64→2)` softmax MLP. It is fed by the
    **post-cross-attention** mean-pooled summaries, which are D's own `z_a` and `z_s`, the vectors that also feed
    InfoNCE and KL. It scales `H̃_a` and `H̃_s` before `[g_a·H̃_a ; SEP ; g_i·H̃_s]` enters Mamba. The SEP token is
    not gated.
- **5d:** parameter table (8 models) and structural hooks.

In [ ]:
# ==============================================================================
# 5a. CONDITION D — v24's model + loss classes, VERBATIM (v22 == v24 == v27); build_fusion_blocks: real mamba_ssm only
# ==============================================================================
import math


def build_fusion_blocks(d_model, num_blocks, d_state, d_conv, expand=MAMBA_EXPAND):
    """v24's function without the CPU fallback branch: real mamba_ssm.Mamba blocks only."""
    return nn.ModuleList([MambaBlock(d_model, d_state, d_conv, expand) for _ in range(num_blocks)])


# ---- VERBATIM from version 24/minor-project-v24.ipynb (model cell + AeroSyncLoss from the training cell) ----
# sha1[:10] of each source (identical in v22, v24, v27): ContinuousTimeEmbedding 6eb422a2ea, HeterogeneousTemporalTokenization f378aaef09, GridLayerNorm 3a64dba834, PairwiseLACA da33fc4f61, BiModalLACAFusion 567cf3d5d6, MambaBlock b88a5bf138, AeroSyncMamba 9d843e21a5, verify_architecture a7c78390a9, AeroSyncLoss 468046c494

class ContinuousTimeEmbedding(nn.Module):
    def __init__(self, d_model: int, num_frequencies: int = TIME_EMBED_FREQUENCIES):
        super().__init__()
        self.frequencies = nn.Parameter(
            torch.exp(torch.linspace(0, math.log(1000.0), num_frequencies)), requires_grad=False
        )
        self.mlp = nn.Sequential(
            nn.Linear(2 * num_frequencies, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )

    def forward(self, tau: torch.Tensor) -> torch.Tensor:
        if tau.dim() == 2:
            tau = tau.unsqueeze(-1)
        angles = tau * self.frequencies * 2.0 * math.pi
        return self.mlp(torch.cat([torch.sin(angles), torch.cos(angles)], dim=-1))


class HeterogeneousTemporalTokenization(nn.Module):
    """token = LayerNorm(features + modality embedding [+ time embedding(tau)]); modality 0 = audio, 1 = sensor.
    v22: time_enabled = HTT_TIME_EMBED_ENABLED. When False the time embedding is skipped entirely (no temporal
    position information enters through HTT); the module is still built so every condition draws the same RNG."""

    def __init__(self, d_model: int, time_enabled: bool = True):
        super().__init__()
        self.d_model = d_model
        self.time_enabled = time_enabled
        self.modality_embeddings = nn.Embedding(2, d_model)
        self.time_embedding = ContinuousTimeEmbedding(d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(self, z: torch.Tensor, tau: torch.Tensor, modality_idx: int) -> torch.Tensor:
        mod_idx = torch.tensor(modality_idx, device=z.device, dtype=torch.long)
        e_m = self.modality_embeddings(mod_idx).view(1, 1, self.d_model)
        if not self.time_enabled:
            return self.norm(z + e_m)                       # HTT_TIME_EMBED_ENABLED = False
        return self.norm(z + e_m + self.time_embedding(tau))


class GridLayerNorm(nn.Module):
    """LayerNorm for the lag-bias grid (v18).

    g(Δτ) produces ONE scalar per (query i, key j) pair, so its output is a (B, T_m, T_n) grid with no feature
    dimension > 1. A LayerNorm over a size-1 dimension would subtract each value from itself and return 0 for every
    input. Instead, the normalization is taken over each window's whole (T_m, T_n) grid, i.e. every bias value that
    enters that window's attention matrix. It is computed separately for each window: every window has its own real
    timestamps, and mixing windows would make one window's bias depend on the rest of the batch and differ between
    training and evaluation. There are no affine parameters, so beta is the only source of scale.

    Consequences: the subtracted mean is one constant per window, and softmax is unchanged by a constant added to
    every score, so only the division by the std affects attention. For an all-zero grid (the zero-init state) the
    output is exactly 0. As g develops variance v, the output std grows smoothly as sqrt(v / (v + eps)) toward 1."""

    def __init__(self, eps: float = LAG_BIAS_LN_EPS):
        super().__init__()
        self.eps = eps

    def forward(self, x: torch.Tensor) -> torch.Tensor:   # x: (B, T_m, T_n)
        return F.layer_norm(x, x.shape[-2:], eps=self.eps)


class PairwiseLACA(nn.Module):
    """Cross-attention from modality m to n with a learned lag bias g(Δτ) added to the scores (scaled by β)."""

    def __init__(self, d_model: int, num_heads: int = LACA_NUM_HEADS, beta: float = LACA_BETA):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.scale = 1.0 / math.sqrt(self.head_dim)
        self.beta = beta
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.lag_bias = nn.Sequential(nn.Linear(1, LAG_BIAS_HIDDEN), nn.Tanh(), nn.Linear(LAG_BIAS_HIDDEN, 1))
        # v18 ZERO-INIT: g's final Linear starts at exactly 0 -> g(Δτ) = 0 for every Δτ and every seed. The layer was
        # already built (same RNG draws as v16), and nn.init.zeros_ draws no random numbers, so every other
        # layer's initial weights and the training RNG stream are unchanged.
        nn.init.zeros_(self.lag_bias[-1].weight)
        nn.init.zeros_(self.lag_bias[-1].bias)
        self.lag_bias_norm = GridLayerNorm()   # v18: no parameters

    def lag_bias_term(self, tau_m, tau_n):
        """LN_grid(g(Δτ)), BEFORE beta. Returns (B, 1, T_m, T_n), broadcast over the heads."""
        delta_tau = (tau_m.unsqueeze(2) - tau_n.unsqueeze(1)).unsqueeze(-1)   # (B, T_m, T_n, 1)
        g = self.lag_bias(delta_tau).squeeze(-1)                                # (B, T_m, T_n): one scalar per (i, j)
        return self.lag_bias_norm(g).unsqueeze(1)                               # normalized per window -> (B, 1, T_m, T_n)

    def forward(self, H_m, H_n, tau_m, tau_n):
        B, T_m, _ = H_m.shape
        _, T_n, _ = H_n.shape
        Q = self.w_q(H_m).view(B, T_m, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.w_k(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.w_v(H_n).view(B, T_n, self.num_heads, self.head_dim).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale
        lag_bias = self.lag_bias_term(tau_m, tau_n)

        attn = F.softmax(scores + self.beta * lag_bias, dim=-1)   # beta = 1.0 (lag bias ON) or exactly 0.0 (OFF)
        out = torch.matmul(attn, V).transpose(1, 2).contiguous().view(B, T_m, self.d_model)

        attn_mean = attn.mean(dim=1)
        tau_diff = tau_n.unsqueeze(1) - tau_m.unsqueeze(2)
        expected_lag = (attn_mean * tau_diff).sum(dim=(1, 2)) / float(T_m)
        return self.out_proj(out), expected_lag


class BiModalLACAFusion(nn.Module):
    def __init__(self, d_model: int, beta: float = LACA_BETA):
        super().__init__()
        self.laca_as = PairwiseLACA(d_model, beta=beta)   # beta scales the lag bias in BOTH directions
        self.laca_sa = PairwiseLACA(d_model, beta=beta)
        self.gamma_as = nn.Parameter(torch.tensor(0.5))
        self.gamma_sa = nn.Parameter(torch.tensor(0.5))

    def forward(self, Ha, Hs, tau_a, tau_s):
        H_a_from_s, lag_as = self.laca_as(Ha, Hs, tau_a, tau_s)
        H_s_from_a, lag_sa = self.laca_sa(Hs, Ha, tau_s, tau_a)
        return Ha + self.gamma_as * H_a_from_s, Hs + self.gamma_sa * H_s_from_a, {"lag_as": lag_as, "lag_sa": lag_sa}


class MambaBlock(nn.Module):
    """Pre-norm residual wrapper around the real mamba_ssm.Mamba selective-scan SSM."""

    def __init__(self, d_model: int, d_state: int, d_conv: int, expand: int):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.mamba = MAMBA_CLASS(d_model=d_model, d_state=d_state, d_conv=d_conv, expand=expand)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.mamba(self.norm(x))


# The reference results were produced by a model whose constructor drew, from the torch RNG, the initial weights
# of a 4-block (d_state 16, d_conv 4) Mamba stack before the final 2-block stack was built from the SAME starting
# RNG state. Every module built afterwards (pooling, classifier, auxiliary heads) and the whole training RNG stream
# (shuffle order, dropout masks) depend on the RNG state that 4-block draw leaves behind. To reproduce those
# results exactly, the constructor performs the same draw and discards it; the discarded blocks are never part of
# the model. This is a fixed step of weight initialization, not a configuration option.
RNG_REFERENCE_STACK = (4, 16, 4)   # (num_blocks, d_state, d_conv) whose initialization is drawn and discarded


class AeroSyncMamba(nn.Module):
    def __init__(self, d_model: int = D_MODEL, n_classes: int = N_CLASSES, htt_time: bool = True, laca_lag: bool = False):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.htt_time_enabled = htt_time                  # HTT_TIME_EMBED_ENABLED
        self.laca_lag_enabled = laca_lag                  # LACA_LAG_BIAS_ENABLED
        self.laca_beta = LACA_BETA if laca_lag else 0.0   # OFF = exactly 0.0 (v18's beta = 0 code path)

        self.sensor_encoder = nn.Sequential(
            nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, d_model, kernel_size=3, padding=1), nn.BatchNorm1d(d_model), nn.GELU(),
        )
        self.audio_encoder = nn.Sequential(
            nn.Linear(N_AUDIO_BINS, d_model), nn.LayerNorm(d_model), nn.GELU(), nn.Linear(d_model, d_model)
        )
        self.htt = HeterogeneousTemporalTokenization(d_model, time_enabled=htt_time)
        self.laca = BiModalLACAFusion(d_model, beta=self.laca_beta)   # plain float: no params, no RNG draws
        self.sep_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)

        rng_before = torch.get_rng_state()
        build_fusion_blocks(d_model, *RNG_REFERENCE_STACK)
        rng_after_reference = torch.get_rng_state()
        torch.set_rng_state(rng_before)
        self.mamba_blocks = build_fusion_blocks(d_model, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        torch.set_rng_state(rng_after_reference)

        self.pool_attn = nn.Sequential(nn.Linear(d_model, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(
            nn.Dropout(DROPOUT),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Linear(d_model // 2, n_classes),
        )
        self.aux_a = nn.Linear(d_model, n_classes)
        self.aux_s = nn.Linear(d_model, n_classes)

    def forward(self, xa, xs, ta, ts):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        F_seq = torch.cat([H_tilde_a, self.sep_token.expand(B, -1, -1), H_tilde_s], dim=1)
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, lags


def verify_architecture(model):
    """Finalized architecture, read from the constructed modules, including both v22 switches."""
    assert model.htt.time_enabled == model.htt_time_enabled
    beta = LACA_BETA if model.laca_lag_enabled else 0.0
    for name in ("laca_as", "laca_sa"):
        laca = getattr(model.laca, name)
        assert laca.num_heads == LACA_NUM_HEADS, f"{name}: {laca.num_heads} heads"
        assert laca.beta == beta, f"{name}: beta {laca.beta} != {beta} for LACA_LAG_BIAS_ENABLED={model.laca_lag_enabled}"
        assert sum(p.numel() for p in laca.lag_bias.parameters()) == LAG_BIAS_PARAMS
        assert isinstance(laca.lag_bias_norm, GridLayerNorm) and not list(laca.lag_bias_norm.parameters())
    assert len(model.mamba_blocks) == MAMBA_NUM_BLOCKS
    mamba_params = sum(p.numel() for p in model.mamba_blocks.parameters())
    if MAMBA_CLASS is not None:
        for block in model.mamba_blocks:
            m = block.mamba
            assert (m.d_state, m.d_conv, m.expand) == (MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND)
        assert mamba_params == EXPECTED_MAMBA_PARAMS, f"Mamba params {mamba_params:,} != {EXPECTED_MAMBA_PARAMS:,}"
    return mamba_params


class AeroSyncLoss(nn.Module):
    """L = CE_classweighted(logits, y) + LAMBDA_ALIGN * InfoNCE(z_a, z_s) + LAMBDA_CONS * KL-consistency."""

    def __init__(self, class_weights):
        super().__init__()
        self.lambda_align = LAMBDA_ALIGN
        self.lambda_cons = LAMBDA_CONS
        self.temperature = INFONCE_TEMPERATURE
        self.ce = nn.CrossEntropyLoss(weight=class_weights)

    def _infonce(self, z1, z2):
        sim = torch.matmul(F.normalize(z1, p=2, dim=-1), F.normalize(z2, p=2, dim=-1).t()) / self.temperature
        labels = torch.arange(z1.shape[0], device=z1.device)
        return 0.5 * (F.cross_entropy(sim, labels) + F.cross_entropy(sim.t(), labels))

    def forward(self, logits, targets, pooled, aux_logits):
        loss_cls = self.ce(logits, targets)
        loss_align = self._infonce(pooled["z_a"], pooled["z_s"])
        p_fused = F.softmax(logits.detach(), dim=-1)
        loss_cons = (
            F.kl_div(F.log_softmax(aux_logits["y_a"], dim=-1), p_fused, reduction="batchmean") +
            F.kl_div(F.log_softmax(aux_logits["y_s"], dim=-1), p_fused, reduction="batchmean")
        ) / 2.0
        total = loss_cls + self.lambda_align * loss_align + self.lambda_cons * loss_cons
        return total, {"loss_cls": loss_cls.item(), "loss_align": loss_align.item(), "loss_cons": loss_cons.item()}

In [ ]:
# ==============================================================================
# 5b. A / I (v28 UnimodalMamba) and the NEW encoders + F0 / F1 / F2
# ==============================================================================
class UnimodalMamba(nn.Module):
    """v28's single-modality path: encoder -> + modality embedding -> LayerNorm -> 2 Mamba blocks -> attention pooling
    -> classifier. Reads only its own modality from the shared paired windows."""

    def __init__(self, modality):
        super().__init__()
        self.modality = modality
        if modality == "audio":
            self.audio_encoder = nn.Sequential(nn.Linear(N_AUDIO_FEAT, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(),
                                               nn.Linear(D_MODEL, D_MODEL))
        else:
            self.sensor_encoder = nn.Sequential(
                nn.Conv1d(N_SENSOR_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
                nn.Conv1d(64, D_MODEL, kernel_size=3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())
        self.modality_embedding = nn.Embedding(1, D_MODEL)
        self.token_norm = nn.LayerNorm(D_MODEL)
        self.mamba_blocks = build_fusion_blocks(D_MODEL, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))

    def forward(self, xa, xs, ta, ts, gate_override=None):
        z = self.audio_encoder(xa) if self.modality == "audio" else self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)
        h = self.token_norm(z + self.modality_embedding.weight.view(1, 1, -1))
        for block in self.mamba_blocks:
            h = block(h)
        w = F.softmax(self.pool_attn(h), dim=1)
        return self.classifier((h * w).sum(dim=1)), None, None, {}


class GatedAudioEncoder(nn.Module):
    """NEW audio encoder: ONE learnable elementwise feature mask m = sigmoid(w), w in R^80 (a fixed learned mask, NOT
    input-dependent), then the existing audio projection."""

    def __init__(self):
        super().__init__()
        self.gate_logit = nn.Parameter(torch.zeros(N_AUDIO_FEAT))      # m = 0.5 at init; zeros_ draws no RNG
        self.proj = nn.Sequential(nn.Linear(N_AUDIO_FEAT, D_MODEL), nn.LayerNorm(D_MODEL), nn.GELU(), nn.Linear(D_MODEL, D_MODEL))

    def mask(self):
        return torch.sigmoid(self.gate_logit)

    def forward(self, xa):
        return self.proj(self.mask() * xa)


class ConvIMUEncoder(nn.Module):
    """NEW IMU encoder: ONE Conv1d(6 -> 6, k=7, 'same' padding) before the existing IMU projection (unchanged)."""

    def __init__(self):
        super().__init__()
        self.pre_conv = nn.Conv1d(N_SENSOR_CHANNELS, IMU_PRECONV_CHANNELS, kernel_size=IMU_PRECONV_KERNEL,
                                  padding=IMU_PRECONV_KERNEL // 2)
        self.proj = nn.Sequential(
            nn.Conv1d(IMU_PRECONV_CHANNELS, 64, kernel_size=5, padding=2), nn.BatchNorm1d(64), nn.GELU(),
            nn.Conv1d(64, D_MODEL, kernel_size=3, padding=1), nn.BatchNorm1d(D_MODEL), nn.GELU())

    def forward(self, xs):                                  # (B, T, 6) -> (B, T, D)
        return self.proj(self.pre_conv(xs.transpose(1, 2))).transpose(1, 2)


class GatedFusionMamba(nn.Module):
    """F0: [A ; I] (no gate). F1: softmax gate [g_a, g_i]. F2: two independent sigmoid gates.
    A_summary / I_summary (mean pooling, as D) feed the gate AND the InfoNCE / KL losses (no second pooled vector)."""

    def __init__(self, kind):
        super().__init__()
        assert kind in ("F0", "F1", "F2")
        self.kind = kind
        self.audio_encoder = GatedAudioEncoder()
        self.sensor_encoder = ConvIMUEncoder()
        self.modality_embeddings = nn.Embedding(2, D_MODEL)          # D's HTT with the time embedding OFF
        self.token_norm = nn.LayerNorm(D_MODEL)
        if kind == "F1":
            self.gate_mlp = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 2))
        elif kind == "F2":
            self.gate_a = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 1))
            self.gate_i = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 1))
        self.mamba_blocks = build_fusion_blocks(D_MODEL, MAMBA_NUM_BLOCKS, MAMBA_D_STATE, MAMBA_D_CONV)
        self.pool_attn = nn.Sequential(nn.Linear(D_MODEL, 64), nn.Tanh(), nn.Linear(64, 1))
        self.classifier = nn.Sequential(nn.Dropout(DROPOUT), nn.Linear(D_MODEL, D_MODEL // 2), nn.GELU(),
                                        nn.Linear(D_MODEL // 2, N_CLASSES))
        self.aux_a = nn.Linear(D_MODEL, N_CLASSES)
        self.aux_s = nn.Linear(D_MODEL, N_CLASSES)

    def forward(self, xa, xs, ta, ts, gate_override=None):
        B = xa.shape[0]
        e = self.modality_embeddings.weight
        A = self.token_norm(self.audio_encoder(xa) + e[0].view(1, 1, -1))     # (B, 64, D)
        I = self.token_norm(self.sensor_encoder(xs) + e[1].view(1, 1, -1))    # (B, 128, D)
        a_sum, i_sum = A.mean(dim=1), I.mean(dim=1)
        s = torch.cat([a_sum, i_sum], dim=-1)
        if self.kind == "F0":
            g_a = g_i = torch.ones(B, device=xa.device)
        elif self.kind == "F1":
            g = F.softmax(self.gate_mlp(s), dim=-1)
            g_a, g_i = g[:, 0], g[:, 1]
        else:
            g_a, g_i = torch.sigmoid(self.gate_a(s)).squeeze(-1), torch.sigmoid(self.gate_i(s)).squeeze(-1)
        if gate_override is not None:                                          # diagnostics only: constant gates
            g_a = torch.full((B,), float(gate_override[0]), device=xa.device)
            g_i = torch.full((B,), float(gate_override[1]), device=xa.device)
        H = torch.cat([g_a.view(B, 1, 1) * A, g_i.view(B, 1, 1) * I], dim=1)   # (B, 192, D)
        for block in self.mamba_blocks:
            H = block(H)
        w = F.softmax(self.pool_attn(H), dim=1)
        logits = self.classifier((H * w).sum(dim=1))
        return (logits, {"z_a": a_sum, "z_s": i_sum}, {"y_a": self.aux_a(a_sum), "y_s": self.aux_s(i_sum)},
                {"g_a": g_a, "g_i": g_i})


class DWrapper(nn.Module):
    """Condition D with the common interface; the wrapped AeroSyncMamba is v24's class, unmodified."""

    def __init__(self):
        super().__init__()
        self.net = AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED)
        self.mamba_blocks = self.net.mamba_blocks

    def forward(self, xa, xs, ta, ts, gate_override=None):
        logits, pooled, aux, lags = self.net(xa, xs, ta, ts)
        return logits, pooled, aux, {}


def build_model(name):
    if name == "A":
        return UnimodalMamba("audio")
    if name == "I":
        return UnimodalMamba("sensor")
    if name == "D":
        return DWrapper()
    return GatedFusionMamba(name)

In [ ]:
# ==============================================================================
# 5c. CA and CAG — D's AeroSyncMamba with v29's NEW encoders (CA), plus F1's gate after cross-attention (CAG)
# ==============================================================================
class ConvIMUEncoderCT(ConvIMUEncoder):
    """v29's ConvIMUEncoder (same layers, same parameters, same initialization order). D's forward calls
    sensor_encoder(xs.transpose(1, 2)).transpose(1, 2), i.e. passes (B, 6, T) and expects (B, D, T) back; the only
    difference from v29's forward is that this one skips v29's internal transposes."""

    def forward(self, x_ct):
        return self.proj(self.pre_conv(x_ct))


class CrossAttnMamba(AeroSyncMamba):
    """CA: v24's AeroSyncMamba (Condition D: HTT time OFF, LACA beta 0.0, [H_a ; SEP ; H_s], 2 Mamba blocks, attention
    pooling, classifier, aux heads) with ONLY its two encoders replaced by v29's new encoders. forward is inherited
    from AeroSyncMamba unchanged."""

    def __init__(self):
        super().__init__(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED)
        self.audio_encoder = GatedAudioEncoder()
        self.sensor_encoder = ConvIMUEncoderCT()


class CrossAttnGateMamba(CrossAttnMamba):
    """CAG: CA + F1's softmax content gate AFTER cross-attention, BEFORE Mamba. The lines not marked 'CAG' are
    AeroSyncMamba.forward's own lines."""

    def __init__(self):
        super().__init__()
        self.gate_mlp = nn.Sequential(nn.Linear(2 * D_MODEL, GATE_HIDDEN), nn.GELU(), nn.Linear(GATE_HIDDEN, 2))   # F1's MLP

    def forward(self, xa, xs, ta, ts, gate_override=None):
        B = xa.shape[0]
        za = self.audio_encoder(xa)
        zs = self.sensor_encoder(xs.transpose(1, 2)).transpose(1, 2)

        Ha = self.htt(za, ta, modality_idx=0)
        Hs = self.htt(zs, ts, modality_idx=1)
        H_tilde_a, H_tilde_s, lags = self.laca(Ha, Hs, ta, ts)
        za_p = H_tilde_a.mean(dim=1)
        zs_p = H_tilde_s.mean(dim=1)

        g = F.softmax(self.gate_mlp(torch.cat([za_p, zs_p], dim=-1)), dim=-1)          # CAG: gate from post-x-attn summaries
        g_a, g_i = g[:, 0], g[:, 1]                                                     # CAG
        if gate_override is not None:                                                   # CAG (diagnostics only)
            g_a = torch.full((B,), float(gate_override[0]), device=xa.device)
            g_i = torch.full((B,), float(gate_override[1]), device=xa.device)
        F_seq = torch.cat([g_a.view(B, 1, 1) * H_tilde_a, self.sep_token.expand(B, -1, -1),
                           g_i.view(B, 1, 1) * H_tilde_s], dim=1)                        # CAG: gated streams, SEP ungated
        for block in self.mamba_blocks:
            F_seq = block(F_seq)

        weights = F.softmax(self.pool_attn(F_seq), dim=1)
        zf = (F_seq * weights).sum(dim=1)
        logits = self.classifier(zf)

        pooled = {"z_a": za_p, "z_s": zs_p}
        aux_logits = {"y_a": self.aux_a(za_p), "y_s": self.aux_s(zs_p)}
        return logits, pooled, aux_logits, {"g_a": g_a, "g_i": g_i}


class NetWrapper(nn.Module):
    """Common interface (logits, pooled, aux, extras) for D / CA / CAG."""

    def __init__(self, net):
        super().__init__()
        self.net = net
        self.mamba_blocks = net.mamba_blocks

    def forward(self, xa, xs, ta, ts, gate_override=None):
        if isinstance(self.net, CrossAttnGateMamba):
            return self.net(xa, xs, ta, ts, gate_override=gate_override)
        logits, pooled, aux, _ = self.net(xa, xs, ta, ts)
        return logits, pooled, aux, {}


def build_model_v30(name):
    if name == "CA":
        return NetWrapper(CrossAttnMamba())
    if name == "CAG":
        return NetWrapper(CrossAttnGateMamba())
    if name == "D":
        return NetWrapper(AeroSyncMamba(htt_time=HTT_TIME_EMBED_ENABLED, laca_lag=LACA_LAG_BIAS_ENABLED))
    return build_model(name)                                     # v29's builder (A, I, F0, F1, F2) — structure checks only

In [ ]:
# ==============================================================================
# 5d. PARAMETER TABLE (8 models) + STRUCTURAL HOOKS
# ==============================================================================
seed_all(42)
_probe = {n: build_model_v30(n) for n in ["D", "F1", "CA", "CAG"]}
N_PARAMS = {n: sum(p.numel() for p in m.parameters()) for n, m in _probe.items()}


def _module_signature(mod):
    return [(n, type(m).__name__) for n, m in mod.named_modules()], [(n, tuple(p.shape)) for n, p in mod.named_parameters()]


# (1) CA's cross-attention is D's own mechanism: same classes, same module tree, same parameter shapes, inherited forward
d_net, ca_net, cag_net = _probe["D"].net, _probe["CA"].net, _probe["CAG"].net
_same_laca = (type(ca_net.laca) is type(d_net.laca) is BiModalLACAFusion and type(cag_net.laca) is BiModalLACAFusion
              and _module_signature(ca_net.laca) == _module_signature(d_net.laca) == _module_signature(cag_net.laca)
              and type(ca_net.htt) is type(d_net.htt) and _module_signature(ca_net.htt) == _module_signature(d_net.htt))
_same_forward = CrossAttnMamba.forward is AeroSyncMamba.forward
_rest_same = all(_module_signature(getattr(ca_net, a)) == _module_signature(getattr(d_net, a))
                 for a in ("pool_attn", "classifier", "aux_a", "aux_s", "mamba_blocks")) and ca_net.sep_token.shape == d_net.sep_token.shape
print("Module diff D -> CA (names present in only one of them):")
_dn = {n for n, _ in d_net.named_modules()}
_cn = {n for n, _ in ca_net.named_modules()}
print(f"   only in D : {sorted(_dn - _cn) or 'none'}\n   only in CA: {sorted(_cn - _dn) or 'none'}")
hook("structure", _same_laca and _same_forward and _rest_same,
     "CA / CAG use D's BiModalLACAFusion (identical module tree and parameter shapes), D's HTT, SEP token, Mamba stack, "
     "pooling, classifier and aux heads; CA.forward IS AeroSyncMamba.forward (inherited) — only the encoders differ")
# (2) lag bias exactly zero and no time embedding, as in D
_lag0 = all(getattr(n_.laca, d).beta == 0.0 for n_ in (d_net, ca_net, cag_net) for d in ("laca_as", "laca_sa"))
_htt0 = not any(n_.htt.time_enabled for n_ in (d_net, ca_net, cag_net))
hook("no-timestamps", _lag0 and _htt0, f"LACA beta = 0.0 in both directions and HTT time embedding OFF for D, CA, CAG")
# (3) functional check on an untrained probe: predictions do not change when every timestamp is replaced
torch.manual_seed(0)
_xa = torch.randn(8, TA_AUDIO, N_AUDIO_FEAT, device=device)
_xs = torch.randn(8, TS_SENSOR, N_SENSOR_CHANNELS, device=device)
_ta, _ts = torch.rand(8, TA_AUDIO, device=device), torch.rand(8, TS_SENSOR, device=device)
_inv = True
with torch.no_grad():
    for n in ("CA", "CAG"):
        m = _probe[n].to(device).eval()
        _inv &= torch.equal(m(_xa, _xs, _ta, _ts)[0], m(_xa, _xs, torch.zeros_like(_ta), torch.randn_like(_ts) * 100)[0])
        m.cpu()
hook("no-timestamps", _inv, "untrained CA/CAG: logits bit-identical when every tau is replaced (zeros / random) — no "
                            "timestamp information reaches the output")
# (4) CAG's gate MLP is F1's exact gate MLP
_f1_gate = _probe["F1"].gate_mlp
hook("structure", _module_signature(cag_net.gate_mlp) == _module_signature(_f1_gate) and repr(cag_net.gate_mlp) == repr(_f1_gate),
     f"CAG gate MLP == F1's: {repr(_f1_gate).replace(chr(10), ' ')}")
# (5) real mamba_ssm, same config as every other model
_mcfg = {n: (tuple(type(b.mamba) for b in m.mamba_blocks), tuple((b.mamba.d_state, b.mamba.d_conv, b.mamba.expand) for b in m.mamba_blocks),
             sum(p.numel() for p in m.mamba_blocks.parameters())) for n, m in _probe.items()}
hook("structure", all(all(t is MAMBA_CLASS for t in c[0]) and c[1] == ((MAMBA_D_STATE, MAMBA_D_CONV, MAMBA_EXPAND),) * MAMBA_NUM_BLOCKS
                      and c[2] == EXPECTED_MAMBA_PARAMS for c in _mcfg.values()),
     f"D, F1, CA, CAG: {MAMBA_NUM_BLOCKS} x {MAMBA_CLASS.__module__}.{MAMBA_CLASS.__name__} (d_state {MAMBA_D_STATE}, d_conv "
     f"{MAMBA_D_CONV}, expand {MAMBA_EXPAND}, {EXPECTED_MAMBA_PARAMS:,} params)")
_enc_added = sum(p.numel() for p in ca_net.audio_encoder.parameters()) + sum(p.numel() for p in ca_net.sensor_encoder.parameters()) \
    - sum(p.numel() for p in d_net.audio_encoder.parameters()) - sum(p.numel() for p in d_net.sensor_encoder.parameters())
print(f"CA - D parameters = {N_PARAMS['CA'] - N_PARAMS['D']:,} (new encoders add {_enc_added}: audio gate 80 + IMU pre-Conv1d 258); "
      f"CAG - CA = {N_PARAMS['CAG'] - N_PARAMS['CA']:,} (F1's gate MLP)")
del _probe, d_net, ca_net, cag_net
torch.cuda.empty_cache()
# the 8-model parameter table is printed in section 9 (v29's counts are read from its results CSV)

## 6. Training
v29's procedure and helper functions, verbatim: `compute_metrics`, `fold_arrays`, `make_loader`, `predict` and
`mission_level`.
- **Loss:** v24's `AeroSyncLoss` (class-weighted CE + 0.1·InfoNCE(z_a, z_s) + 0.1·KL) on D's own pooled
  post-cross-attention vectors.
- **Optimizer and schedule:** AdamW 1e-4 with weight decay 1e-4, cosine annealing to 1e-6, clipping at 1.0,
  20 epochs, batch 32, best-validation-macro-F1 checkpoint.
- **Seeding:** `seed_all(seed)` at the start of every run, a fresh model, and a seeded DataLoader generator.
- **Cleanup:** GPU memory is freed after each run.

**Extra per-run diagnostics (CA, CAG):**
- **Cross-attention weights.** On 64 evenly spaced test windows, both directions' 4-head attention matrices are
  recomputed from LACA's own layers and inputs, which are captured with a forward hook. The recomputation uses the
  exact formula `softmax(QKᵀ/√d + β·lag_bias)` with β = 0.
- **Summary statistics:** mean, SD, min, max, mean normalized row entropy (1.0 = uniform), mean max-weight ÷ uniform
  weight, the across-window SD of the head-averaged map, and the final learned residual scales γ_as and γ_sa.
- **Timestamp invariance.** Test predictions are recomputed with every tau set to zero and must be bit-identical.
- **CAG gates.** The g_a / g_i distribution and the constant-gate effect, as for F1 in v29.

**Determinism is not re-investigated** (see the header). The flags from v29 stay set.

In [ ]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 18): compute_metrics, fold_arrays, make_loader, predict, mission_level
# ==============================================================================
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, roc_auc_score)
from scipy import stats
LABELS = list(range(N_CLASSES))


def compute_metrics(y, P):
    pred = P.argmax(1)
    try:
        auroc = float(roc_auc_score(y, P, multi_class="ovr", average="macro", labels=LABELS))
    except ValueError:
        auroc = float("nan")
    return {"accuracy": float(accuracy_score(y, pred)),
            "macro_f1": float(f1_score(y, pred, average="macro", labels=LABELS, zero_division=0)),
            "balanced_accuracy": float(balanced_accuracy_score(y, pred)), "macro_auroc": auroc}


def fold_arrays(k, split):
    P_, idx = FOLD_PREP[k], FOLD_PREP[k]["idx"][split]
    return (((XA[idx] - P_["a_mu"]) / P_["a_sd"]).astype(np.float32), ((XS[idx] - P_["s_mu"]) / P_["s_sd"]).astype(np.float32),
            TAU_A[idx], TAU_S[idx], YW[idx])


def make_loader(arrs, shuffle, seed):
    ds = TensorDataset(*[torch.from_numpy(np.ascontiguousarray(a)) for a in arrs])
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=True,
                      worker_init_fn=worker_init_fn, generator=make_generator(seed) if shuffle else None)


@torch.no_grad()
def predict(model, loader, gate_override=None):
    model.eval()
    probs, ga, gi = [], [], []
    for xa, xs, ta, ts, _ in loader:
        logits, _, _, ex = model(xa.to(device), xs.to(device), ta.to(device), ts.to(device), gate_override=gate_override)
        probs.append(F.softmax(logits, dim=-1).float().cpu().numpy())
        if "g_a" in ex:
            ga.append(ex["g_a"].float().cpu().numpy())
            gi.append(ex["g_i"].float().cpu().numpy())
    return np.concatenate(probs), (np.concatenate(ga) if ga else None), (np.concatenate(gi) if gi else None)


def mission_level(missions, y, P):
    df = pd.DataFrame(P, columns=LABELS)
    df["mission"], df["y"] = missions, y
    g = df.groupby("mission", sort=True)
    return g["y"].first().values, g[LABELS].mean().values, g["y"].first().index.values

In [ ]:
# ==============================================================================
# 6b. train_run (v29's, for D-family models) + cross-attention diagnostics
# ==============================================================================
LABELS = list(range(N_CLASSES))


@torch.no_grad()
def attention_diagnostics(model, k):
    """Recompute LACA's attention for ATTN_SAMPLE_WINDOWS evenly spaced test windows of fold k, from LACA's inputs
    (captured by a forward hook) and its own layers — the exact formula in PairwiseLACA.forward."""
    net = model.net
    idx_all = FOLD_PREP[k]["idx"]["test"]
    sel = np.linspace(0, len(idx_all) - 1, min(ATTN_SAMPLE_WINDOWS, len(idx_all))).round().astype(int)
    arrs = [a[sel] for a in fold_arrays(k, "test")]
    captured = {}
    h = net.laca.register_forward_hook(lambda mod, inp, out: captured.__setitem__("inp", inp))
    model.eval()
    maps = {"a2i": [], "i2a": []}
    try:
        for xa, xs, ta, ts, _ in make_loader(arrs, False, 0):
            model(xa.to(device), xs.to(device), ta.to(device), ts.to(device))
            Ha, Hs, tau_a, tau_s = captured["inp"]
            for key, laca, Hm, Hn, tm, tn in (("a2i", net.laca.laca_as, Ha, Hs, tau_a, tau_s),
                                              ("i2a", net.laca.laca_sa, Hs, Ha, tau_s, tau_a)):
                B, Tm, _ = Hm.shape
                Tn = Hn.shape[1]
                Q = laca.w_q(Hm).view(B, Tm, laca.num_heads, laca.head_dim).transpose(1, 2)
                K = laca.w_k(Hn).view(B, Tn, laca.num_heads, laca.head_dim).transpose(1, 2)
                attn = F.softmax(torch.matmul(Q, K.transpose(-2, -1)) * laca.scale + laca.beta * laca.lag_bias_term(tm, tn), dim=-1)
                maps[key].append(attn.float().cpu())
    finally:
        h.remove()
    out = {}
    for key, lst in maps.items():
        A = torch.cat(lst)                                     # (W, heads, Tm, Tn)
        Tn = A.shape[-1]
        ent = -(A.clamp_min(1e-12) * A.clamp_min(1e-12).log()).sum(-1) / math.log(Tn)
        head_mean = A.mean(1)                                  # (W, Tm, Tn)
        out.update({f"attn_{key}_mean": float(A.mean()), f"attn_{key}_std": float(A.std()), f"attn_{key}_min": float(A.min()),
                    f"attn_{key}_max": float(A.max()), f"attn_{key}_entropy_norm": float(ent.mean()),
                    f"attn_{key}_max_over_uniform": float((A.max(-1).values * Tn).mean()),
                    f"attn_{key}_across_window_sd_rel": float(head_mean.std(0).mean() * Tn)})
    out.update({"gamma_as": float(net.laca.gamma_as), "gamma_sa": float(net.laca.gamma_sa)})
    return out


def train_run(name, k, seed, epochs=EPOCHS, save=True, verbose=True):
    t0 = time.time()
    seed_all(seed)                                           # at the START of every run
    model = build_model_v30(name).to(device)
    init_sha1 = hashlib.sha1(b"".join(v.detach().cpu().contiguous().numpy().tobytes()
                                      for v in model.state_dict().values())).hexdigest()[:12]
    assert all(type(b.mamba) is MAMBA_CLASS for b in model.mamba_blocks)
    cw = torch.tensor(FOLD_PREP[k]["class_weights"], device=device)
    criterion = AeroSyncLoss(cw)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR_MIN)
    tr = make_loader(fold_arrays(k, "train"), True, seed)
    va = make_loader(fold_arrays(k, "val"), False, seed)
    te_arrs = fold_arrays(k, "test")
    te = make_loader(te_arrs, False, seed)
    tag = f"[{name} fold {k} ({FOLDS[k]['day']}) seed {seed}]"
    if verbose:
        print(f"{tag} init {init_sha1} | {N_PARAMS[name]:,} params | windows train {len(tr.dataset)} / val "
              f"{len(va.dataset)} / test {len(te.dataset)} | z-score {FOLD_PREP[k]['zhash']}")
    best_f1, best_state, best_epoch = 0.0, None, None
    for epoch in range(1, epochs + 1):
        model.train()
        run, nb = defaultdict(float), 0
        for xa, xs, ta, ts, y in tr:
            xa, xs, ta, ts, y = (t.to(device, non_blocking=True) for t in (xa, xs, ta, ts, y))
            optimizer.zero_grad()
            logits, pooled, aux, _ = model(xa, xs, ta, ts)
            loss, parts = criterion(logits, y, pooled, aux)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=GRAD_CLIP_NORM)
            optimizer.step()
            run["loss"] += loss.item()
            for kk, v in parts.items():
                run[kk] += v
            nb += 1
        scheduler.step()
        assert math.isfinite(run["loss"]), f"{tag} non-finite loss"
        vP, _, _ = predict(model, va)
        vf1 = compute_metrics(va.dataset.tensors[4].numpy(), vP)["macro_f1"]
        improved = vf1 > best_f1
        if improved:
            best_f1, best_state, best_epoch = vf1, copy.deepcopy(model.state_dict()), epoch
        if verbose:
            print(f"{tag} epoch {epoch:02d}/{epochs} | train loss {run['loss'] / nb:.4f} (cls {run['loss_cls'] / nb:.4f}, align "
                  f"{run['loss_align'] / nb:.4f}, cons {run['loss_cons'] / nb:.4f}) | val macro-F1 {vf1:.4f}" + ("  * best" if improved else ""))
    if best_state is None:
        best_state, best_epoch = copy.deepcopy(model.state_dict()), epochs
    model.load_state_dict(best_state)
    res = {"model": name, "fold": k, "fold_day": FOLDS[k]["day"], "seed": seed, "init_sha1": init_sha1, "best_epoch": best_epoch,
           "best_val_macro_f1": round(best_f1, 6), "zstats_sha1": FOLD_PREP[k]["zhash"], "n_params": N_PARAMS[name]}
    if save:
        P, ga, gi = predict(model, te)
        y = te_arrs[4]
        meta = WMETA.iloc[FOLD_PREP[k]["idx"]["test"]]
        my, mP, _ = mission_level(meta.mission.values, y, P)
        res.update({f"win_{kk}": v for kk, v in compute_metrics(y, P).items()})
        res.update({f"mis_{kk}": v for kk, v in compute_metrics(my, mP).items()})
        # timestamp invariance on the trained model: every tau -> 0
        P0, _, _ = predict(model, make_loader((te_arrs[0], te_arrs[1], np.zeros_like(te_arrs[2]), np.zeros_like(te_arrs[3]), te_arrs[4]),
                                              False, seed))
        res["tau_invariant"] = bool(np.array_equal(P, P0))
        res.update(attention_diagnostics(model, k))
        print(f"{tag} TEST window macro-F1 {res['win_macro_f1']:.4f} | mission macro-F1 {res['mis_macro_f1']:.4f} | tau-invariant "
              f"{res['tau_invariant']} | gamma_as {res['gamma_as']:.3f} gamma_sa {res['gamma_sa']:.3f}")
        for key, lab in (("a2i", "audio->IMU"), ("i2a", "IMU->audio")):
            print(f"{tag} ATTENTION {lab:<10} ({ATTN_SAMPLE_WINDOWS} test windows, 4 heads): mean {res[f'attn_{key}_mean']:.5f} sd "
                  f"{res[f'attn_{key}_std']:.5f} min {res[f'attn_{key}_min']:.2e} max {res[f'attn_{key}_max']:.4f} | row entropy "
                  f"{res[f'attn_{key}_entropy_norm']:.4f} of uniform | max/uniform {res[f'attn_{key}_max_over_uniform']:.2f}x | "
                  f"across-window sd {res[f'attn_{key}_across_window_sd_rel']:.3f} x uniform -> "
                  + ("NEAR-UNIFORM" if res[f"attn_{key}_entropy_norm"] > NEAR_UNIFORM_ENTROPY else "content-dependent (non-uniform)"))
        extra = {"audio_mask": model.net.audio_encoder.mask().detach().float().cpu().numpy().astype(np.float32)}
        if name == "CAG":
            P_const, _, _ = predict(model, te, gate_override=(float(ga.mean()), float(gi.mean())))
            res.update({"g_a_mean": float(ga.mean()), "g_a_std": float(ga.std()), "g_a_min": float(ga.min()), "g_a_max": float(ga.max()),
                        "g_i_mean": float(gi.mean()), "g_i_std": float(gi.std()), "g_i_min": float(gi.min()), "g_i_max": float(gi.max()),
                        "gate_effect_mean_abs_dp": float(np.abs(P - P_const).mean()),
                        "gate_effect_argmax_changed": float((P.argmax(1) != P_const.argmax(1)).mean())})
            extra.update({"g_a": ga.astype(np.float32), "g_i": gi.astype(np.float32)})
            print(f"{tag} GATES (test): g_a mean {ga.mean():.3f} sd {ga.std():.3f} [{ga.min():.3f}, {ga.max():.3f}] | g_i mean "
                  f"{gi.mean():.3f} sd {gi.std():.3f} | constant-gate effect: argmax changed {res['gate_effect_argmax_changed']:.3f}")
        fname = f"{name}_{k}_{seed}"
        np.savez(PROBS_DIR / f"{fname}.npz", window_id=meta.window_id.values.astype(str), mission=meta.mission.values.astype(str),
                 day=meta.day.values.astype(str), cls=y, phase=meta.phase.values.astype(str), probs=P.astype(np.float32), **extra)
        torch.save(best_state, CKPT_DIR / f"{fname}.pt")
        res.update({"class_weights": json.dumps(np.round(FOLD_PREP[k]["class_weights"], 4).tolist()),
                    "n_train_windows": len(tr.dataset), "n_val_windows": len(va.dataset), "n_test_windows": len(te.dataset),
                    "probs_file": str(PROBS_DIR / f"{fname}.npz"), "checkpoint": str(CKPT_DIR / f"{fname}.pt")})
    res["train_seconds"] = round(time.time() - t0, 1)
    del model, optimizer, scheduler, best_state, tr, va, te
    gc.collect()
    torch.cuda.empty_cache()
    return res

## 7. Sweep — 2 models × 5 folds × 2 seeds = 20 runs
- **Order:** fold-major, then model (CA, CAG), then seed. A partial session leaves complete CA-vs-CAG comparisons
  for every fold it finished.
- **Logging and resume:** each run is appended to `crossattn_results.csv` immediately. Runs already logged, with their
  `.npz` present, are skipped.
- **Outputs:** probabilities go to `v30_probs/{model}_{fold}_{seed}.npz` (v29's schema, plus `audio_mask`, and
  `g_a`/`g_i` for CAG), and checkpoints to `v30_checkpoints/`.

In [ ]:
# ==============================================================================
# 7. SWEEP (resumable)
# ==============================================================================
ATTN_COLS = [f"attn_{d}_{s}" for d in ("a2i", "i2a") for s in ("mean", "std", "min", "max", "entropy_norm", "max_over_uniform",
                                                             "across_window_sd_rel")]
CSV_FIELDS = ["model", "fold", "fold_day", "seed", "n_train_windows", "n_val_windows", "n_test_windows",
              "win_accuracy", "win_macro_f1", "win_balanced_accuracy", "win_macro_auroc",
              "mis_accuracy", "mis_macro_f1", "mis_balanced_accuracy", "mis_macro_auroc",
              "best_epoch", "best_val_macro_f1", "init_sha1", "zstats_sha1", "class_weights", "n_params", "train_seconds",
              "tau_invariant", "gamma_as", "gamma_sa"] + ATTN_COLS + [
              "g_a_mean", "g_a_std", "g_a_min", "g_a_max", "g_i_mean", "g_i_std", "g_i_min", "g_i_max",
              "gate_effect_mean_abs_dp", "gate_effect_argmax_changed", "probs_file", "checkpoint", "finished_at"]


def load_done():
    if not RESULTS_CSV.exists():
        return {}
    df = pd.read_csv(RESULTS_CSV)
    return {(r.model, int(r.fold), int(r.seed)): r for r in df.itertuples()
            if (PROBS_DIR / f"{r.model}_{int(r.fold)}_{int(r.seed)}.npz").exists()}


def append_row(res):
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, "a", newline="") as f:
        w = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        if new:
            w.writeheader()
        w.writerow({**res, "finished_at": datetime.now().isoformat(timespec="seconds")})


PLAN = [(m, k, s) for k in range(len(DAYS)) for m in V30_MODELS for s in SEEDS]
_done = load_done()
print(f"{len(PLAN)} runs planned | already logged {sum(p in _done for p in PLAN)}")
_t0, _n_new, SWEEP_T0 = time.time(), 0, time.time()
for (m, k, s) in PLAN:
    if (m, k, s) in load_done():
        continue
    res = train_run(m, k, s)
    append_row(res)
    _n_new += 1
    n_complete = sum(p in load_done() for p in PLAN)
    eta = (len(PLAN) - n_complete) * (time.time() - _t0) / _n_new / 60
    print(f"==> {n_complete}/{len(PLAN)} complete | {m} fold {k} seed {s}: window macro-F1 {res['win_macro_f1']:.4f}, "
          f"mission macro-F1 {res['mis_macro_f1']:.4f}, best epoch {res['best_epoch']}, {res['train_seconds']:.0f} s | ETA {eta:.0f} min")
SWEEP_MINUTES = (time.time() - SWEEP_T0) / 60
print(f"Sweep: {sum(p in load_done() for p in PLAN)}/{len(PLAN)} logged | this session trained {_n_new} runs in {SWEEP_MINUTES:.1f} min")

## 8. Load v29's saved results (A, I, D, F0, F1, F2 — NOT retrained)
The following v29 files are read:
- **`gated_fusion_results.csv`:** 60 per-run rows, with window and mission macro-F1 per (model, fold, seed). This
  feeds the **seed-averaged-F1 method**.
- **`v29_results_table.csv`:** the pooled table. Its rows go into the combined table **as they are**, so the metrics
  and the CI method are identical by construction.
- **`v29_probs/*.npz`:** per-window probabilities, needed only for the **pooled-probability method**.

If a model's `.npz` files are incomplete, the pooled-probability method is **unavailable** for every comparison that
involves it. That is printed as a flag, never silently replaced by the other method.

When all of a model's `.npz` files are present, they are checked in two ways:
- their window ids must match this notebook's test folds;
- their pooled macro-F1 must reproduce v29's table value, which proves they are v29's files.

In [ ]:
# ==============================================================================
# 8. LOAD v29 ARTIFACTS
# ==============================================================================
def _expand(pattern):
    p = pattern.replace("{repo}", str(PROJECT_REPO))
    return sorted(glob.glob(p, recursive=True)) if "*" in p else ([p] if Path(p).exists() else [])


def find_first(key):
    for pat in V29_SEARCH[key]:
        hits = _expand(pat)
        if hits:
            return Path(hits[0])
    return None


V29_CSV_PATH, V29_TABLE_PATH = find_first("results_csv"), find_first("table_csv")
if V29_CSV_PATH is None or V29_TABLE_PATH is None:
    loud("v29's gated_fusion_results.csv and/or v29_results_table.csv NOT FOUND. Searched:\n"
         + "\n".join(p.replace("{repo}", str(PROJECT_REPO)) for k in ("results_csv", "table_csv") for p in V29_SEARCH[k])
         + "\nAttach v29's output, or push version 29/Results/ to the project repo, then re-run from section 8 "
           "(the 20 v30 runs are already logged and will not be retrained).")
    raise RuntimeError("v29 results not found — the combined comparison needs them.")
V29_RUNS = pd.read_csv(V29_CSV_PATH)
V29_TABLE = pd.read_csv(V29_TABLE_PATH)
print(f"v29 per-run CSV: {V29_CSV_PATH} ({len(V29_RUNS)} rows) | v29 table: {V29_TABLE_PATH} ({len(V29_TABLE)} rows)")
_keys = {(r.model, int(r.fold), int(r.seed)) for r in V29_RUNS.itertuples()}
hook("v29-reuse", _keys == {(m, k, s) for m in V29_MODELS for k in range(len(DAYS)) for s in SEEDS} and len(V29_RUNS) == 60,
     "v29 CSV holds exactly the 60 runs (6 models x 5 folds x seeds 42/123) — loaded, NOT retrained")
_zh = V29_RUNS.groupby("fold").zstats_sha1.agg(lambda s: set(s))
hook("v29-reuse", all(_zh[k] == {FOLD_PREP[k]["zhash"]} for k in range(len(DAYS))),
     "every v29 run's z-score hash equals this notebook's fold hash (same windows, same normalization)")
hook("v29-reuse", set(V29_TABLE.model) == set(V29_MODELS) and set(V29_TABLE.level) == {"window", "mission"},
     "v29 pooled table has window + mission rows for A, I, D, F0, F1, F2")

# ---- v29 per-window probabilities (pooled-probability method) ----
_NPZ_RE = re.compile(r"^(A|I|D|F0|F1|F2)_(\d)_(\d+)\.npz$")
V29_NPZ, V29_NPZ_SRC = {}, {}
for pat in V29_SEARCH["npz_dirs"]:
    for d in _expand(pat):
        for f in sorted(Path(d).glob("*.npz")):
            m_ = _NPZ_RE.match(f.name)                         # ignores copies such as "I_2_123 (1).npz"
            if m_:
                key = (m_.group(1), int(m_.group(2)), int(m_.group(3)))
                V29_NPZ.setdefault(key, f)
                V29_NPZ_SRC.setdefault(key, d)
NPZ_COMPLETE = {m: all((m, k, s) in V29_NPZ for k in range(len(DAYS)) for s in SEEDS) for m in V29_MODELS}
print("v29 per-window probability files found (of 10 per model): " + ", ".join(
    f"{m} {sum((m, k, s) in V29_NPZ for k in range(len(DAYS)) for s in SEEDS)}" for m in V29_MODELS))
for m in V29_MODELS:
    if not NPZ_COMPLETE[m]:
        missing = [f"{m}_{k}_{s}" for k in range(len(DAYS)) for s in SEEDS if (m, k, s) not in V29_NPZ]
        print(f"   {m}: INCOMPLETE — missing {missing} -> pooled-probability method UNAVAILABLE for comparisons with {m}")

## 9. Combined 8-model table (a direct extension of v29's)
- **The six v29 rows** are loaded from `v29_results_table.csv` unchanged.
- **The CA and CAG rows** are computed here with v29's own functions, copied verbatim: `level_block`,
  `cluster_bootstrap_f1` and `prf_from_cm`. The same probabilities are averaged over seeds 42 and 123, and the same
  mission-level bootstrap is used (2000 resamples of missions within class, seed 2028).

In [ ]:
# ==============================================================================
# v29 helper functions, VERBATIM from version 29/minor-project-v29.ipynb (cell 24): prf_from_cm, cluster_bootstrap_f1, level_block
# ==============================================================================



def prf_from_cm(cm):
    tp = np.diag(cm).astype(float)
    prec = np.divide(tp, cm.sum(0), out=np.zeros_like(tp), where=cm.sum(0) > 0)
    rec = np.divide(tp, cm.sum(1), out=np.zeros_like(tp), where=cm.sum(1) > 0)
    f1 = np.divide(2 * prec * rec, prec + rec, out=np.zeros_like(tp), where=(prec + rec) > 0)
    return prec, rec, f1


def cluster_bootstrap_f1(missions, y, pred, B=BOOTSTRAP_RESAMPLES, seed=BOOTSTRAP_SEED):
    """Macro-F1 over B resamples of MISSIONS within class (never windows)."""
    rng = np.random.default_rng(seed)
    ms = pd.Series(missions).unique()
    pos = {m: i for i, m in enumerate(ms)}
    cnt, mcls = np.zeros((len(ms), N_CLASSES), int), np.zeros(len(ms), int)
    for m, t, p in zip(missions, y, pred):
        cnt[pos[m], p] += 1
        mcls[pos[m]] = t
    cms = np.zeros((B, N_CLASSES, N_CLASSES))
    for c in range(N_CLASSES):
        rows = np.where(mcls == c)[0]
        cms[:, c, :] = cnt[rows[rng.integers(0, len(rows), (B, len(rows)))]].sum(1)
    return np.array([prf_from_cm(cm)[2].mean() for cm in cms])


def level_block(missions, y, P):
    pred = P.argmax(1)
    m = compute_metrics(y, P)
    cm = confusion_matrix(y, pred, labels=LABELS)
    _, rec, _ = prf_from_cm(cm)
    lo, hi = np.percentile(cluster_bootstrap_f1(missions, y, pred), [2.5, 97.5])
    err = pred != y
    return {"accuracy": m["accuracy"], "macro_f1": m["macro_f1"], "f1_ci_low": lo, "f1_ci_high": hi,
            "balanced_accuracy": m["balanced_accuracy"], "macro_auroc": m["macro_auroc"],
            **{f"recall_c{c}": rec[c] for c in range(N_CLASSES)},
            "faulty_pred_normal": float((pred[y > 0] == 0).mean()), "normal_pred_faulty": float((pred[y == 0] > 0).mean()),
            "adjacent_error_share": float((np.abs(pred - y)[err] == 1).mean()) if err.any() else float("nan"), "cm": cm}

In [ ]:
# ==============================================================================
# 9. POOLED TABLE for CA / CAG + COMBINED 8-MODEL TABLE + PARAMETER TABLE
# ==============================================================================
def load_probs(model, k, s):
    f = (PROBS_DIR / f"{model}_{k}_{s}.npz") if model in V30_MODELS else V29_NPZ.get((model, k, s))
    if f is None or not Path(f).exists():
        return None
    z = np.load(f)
    return {key: z[key] for key in z.files}


def pooled_probs(model):
    """Seed-averaged probabilities over all 5 test folds, aligned to window ids (None if any file is missing)."""
    per_seed = []
    for s in SEEDS:
        parts = [load_probs(model, k, s) for k in range(len(DAYS))]
        if any(p is None for p in parts):
            return None
        ids = np.concatenate([p["window_id"] for p in parts])
        o = np.argsort(ids)
        per_seed.append((ids[o], np.concatenate([p["probs"] for p in parts])[o]))
    assert all(np.array_equal(ps[0], per_seed[0][0]) for ps in per_seed)
    ids = per_seed[0][0]
    return ids, WMETA.set_index("window_id").loc[ids].reset_index(), np.mean([ps[1] for ps in per_seed], axis=0)


_done = load_done()
_missing = [p for p in PLAN if p not in _done]
if _missing:
    loud(f"{len(_missing)} of {len(PLAN)} v30 runs are not logged yet: {_missing}\nCA/CAG rows below use only complete models.")
POOLED = {}
rows = []
for m in V30_MODELS:
    pp = pooled_probs(m)
    if pp is None:
        continue
    ids, meta, P = pp
    assert len(ids) == len(WMETA)
    POOLED[m] = (ids, meta, P)
    y = meta.cls.values
    wb = level_block(meta.mission.values, y, P)
    my, mP, mnames = mission_level(meta.mission.values, y, P)
    mb = level_block(mnames, my, mP)
    for lvl, b in (("window", wb), ("mission", mb)):
        rows.append({"model": m, "level": lvl, "seeds": str(SEEDS), "params": N_PARAMS[m], **{k_: v for k_, v in b.items() if k_ != "cm"}})
    print(f"{m} pooled confusion matrix (window level, rows = true):\n{wb['cm']}")

# v29 models whose npz are complete: reproduce v29's pooled macro-F1 (proves the files are v29's)
for m in V29_MODELS:
    if NPZ_COMPLETE[m]:
        pp = pooled_probs(m)
        ids, meta, P = pp
        hook("v29-reuse", np.array_equal(ids, np.sort(WMETA.window_id.values)),
             f"{m}: v29 npz window ids == this notebook's 5238 paired windows", stop=False)
        f_re = compute_metrics(meta.cls.values, P)["macro_f1"]
        f_tab = float(V29_TABLE[(V29_TABLE.model == m) & (V29_TABLE.level == "window")].macro_f1.iloc[0])
        hook("v29-reuse", abs(f_re - f_tab) < 1e-9, f"{m}: pooled window macro-F1 recomputed from npz {f_re:.6f} == v29 table {f_tab:.6f}",
             stop=False)
        POOLED[m] = pp

COMBINED = pd.concat([V29_TABLE, pd.DataFrame(rows)], ignore_index=True)
COMBINED["model"] = pd.Categorical(COMBINED.model, categories=ALL_MODELS, ordered=True)
COMBINED = COMBINED.sort_values(["level", "model"]).reset_index(drop=True)
COMBINED.to_csv(TABLE_CSV, index=False)
pd.set_option("display.width", 260)
for lvl in ("window", "mission"):
    t = COMBINED[COMBINED.level == lvl].set_index("model")
    t["macro-F1 [95% CI]"] = [f"{a:.4f} [{b:.4f}, {c:.4f}]" for a, b, c in zip(t.macro_f1, t.f1_ci_low, t.f1_ci_high)]
    t["source"] = ["v29 (loaded)" if m in V29_MODELS else "v30 (trained)" for m in t.index]
    print(f"\n=== COMBINED POOLED DAY-OUT — {lvl.upper()} LEVEL (8 models, identical protocol and metrics) ===")
    print(t[["source", "params", "accuracy", "macro-F1 [95% CI]", "balanced_accuracy", "macro_auroc"]
            + [f"recall_c{c}" for c in range(N_CLASSES)] + ["faulty_pred_normal", "normal_pred_faulty", "adjacent_error_share"]]
          .round(4).to_string())
print(f"\nsaved: {TABLE_CSV}")

ARCH_V30 = {
    "A": "audio MLP -> 2x Mamba (CE)", "I": "IMU CNN -> 2x Mamba (CE)",
    "D": "old encoders -> LACA x-attn (lag x0) -> [Ha;SEP;Hs] -> 2x Mamba",
    "F0": "new encoders -> [A;I] -> 2x Mamba", "F1": "new encoders -> softmax gate -> [gA;gI] -> 2x Mamba",
    "F2": "new encoders -> 2 sigmoid gates -> 2x Mamba",
    "CA": "NEW encoders -> D's LACA x-attn (lag x0) -> [Ha;SEP;Hs] -> 2x Mamba",
    "CAG": "NEW encoders -> D's LACA -> F1's softmax gate -> [gHa;SEP;gHs] -> 2x Mamba"}
_v29_params = V29_RUNS.groupby("model").n_params.first().to_dict()
print("\nPARAMETER COUNTS (v29 counts from its CSV; D/F1 also re-built here as a check):")
for m in ALL_MODELS:
    n = N_PARAMS[m] if m in V30_MODELS else int(_v29_params[m])
    print(f"   {m:<4} {n:>11,}  {ARCH_V30[m]}")
hook("structure", int(_v29_params["D"]) == N_PARAMS["D"] and int(_v29_params["F1"]) == N_PARAMS["F1"],
     f"D and F1 re-built here have v29's parameter counts ({N_PARAMS['D']:,} / {N_PARAMS['F1']:,})", stop=False)

## 10. Paired comparisons across the 5 folds — BOTH methods, for EVERY comparison
- **Pooled-probability method:** fold macro-F1 of the seed-averaged probabilities. This needs `.npz` files for both
  models.
- **Seed-averaged-F1 method:** the mean over seeds of each run's own fold macro-F1, taken from the per-run CSVs.

For each method the table gives the mean Δ, the SD, wins out of 5, and a paired t-test (df 4), at both window and
mission level.

A **⚠️ METHOD DISAGREEMENT** flag is printed when the two methods give a different win count or a different sign of
the mean Δ. A **⚠️ ONE METHOD ONLY** flag is printed when the `.npz` files needed for the pooled-probability method are
missing.

**Primary comparisons:**
- CA − F0: cross-attention vs plain concatenation, same new encoders (the cleanest test here);
- CA − D: new vs old encoders, both with cross-attention;
- CAG − CA: does a gate after cross-attention help?
- CAG − F1: cross-attention then gate, vs gate alone;
- BEST − I: does anything beat IMU alone?

v29's own four comparisons are **re-checked** under both methods as well.

In [ ]:
# ==============================================================================
# 10. PAIRED FOLD COMPARISONS — pooled-probability method AND seed-averaged-F1 method
# ==============================================================================
ALL_RUNS = pd.concat([V29_RUNS.assign(source="v29"), pd.read_csv(RESULTS_CSV).assign(source="v30")], ignore_index=True)


def fold_scores_seedavg(model, level):
    col = "win_macro_f1" if level == "win" else "mis_macro_f1"
    d = ALL_RUNS[ALL_RUNS.model == model]
    if not all(set(d[d.fold == k].seed) >= set(SEEDS) for k in range(len(DAYS))):
        return None
    return np.array([d[(d.fold == k) & d.seed.isin(SEEDS)][col].mean() for k in range(len(DAYS))])


def fold_scores_pooled(model, level):
    if model not in POOLED:
        return None
    ids, meta, P = POOLED[model]
    out = []
    for k, day in enumerate(DAYS):
        sel = (meta.day == day).values
        y = meta.cls.values[sel]
        if level == "win":
            out.append(compute_metrics(y, P[sel])["macro_f1"])
        else:
            my, mP, _ = mission_level(meta.mission.values[sel], y, P[sel])
            out.append(compute_metrics(my, mP)["macro_f1"])
    return np.array(out)


def pstats(a, b):
    d = a - b
    t, p = stats.ttest_rel(a, b)
    return {"mean": float(d.mean()), "sd": float(d.std(ddof=1)), "wins": int((d > 0).sum()), "t": float(t), "p": float(p),
            "diffs": np.round(d, 4).tolist()}


def verdict_one(s):
    if s["mean"] > NOISE_FLOOR_F1 and s["wins"] >= 4:
        return "BEATS"
    if s["mean"] < -NOISE_FLOOR_F1 and s["wins"] <= 1:
        return "WORSE"
    return "no reliable difference"


COMP_RESULTS = []


def compare(a, b, meaning):
    for level in ("win", "mis"):
        row = {"comparison": f"{a} − {b}", "level": "window" if level == "win" else "mission", "meaning": meaning}
        for method, fn in (("pooled-probability", fold_scores_pooled), ("seed-averaged-F1", fold_scores_seedavg)):
            sa, sb = fn(a, level), fn(b, level)
            row[method] = None if (sa is None or sb is None) else pstats(sa, sb)
        pp, sf = row["pooled-probability"], row["seed-averaged-F1"]
        if pp is None or sf is None:
            row["flag"] = "⚠️ ONE METHOD ONLY (" + ("pooled-probability" if pp is None else "seed-averaged-F1") + " unavailable: npz/rows missing)"
        elif pp["wins"] != sf["wins"] or np.sign(pp["mean"]) != np.sign(sf["mean"]):
            row["flag"] = "⚠️ METHOD DISAGREEMENT (win count or sign differs) — report both"
        else:
            row["flag"] = "methods agree on sign and wins"
        COMP_RESULTS.append(row)
        print(f"\n{row['comparison']} [{row['level']}] — {meaning}")
        for method in ("pooled-probability", "seed-averaged-F1"):
            s = row[method]
            if s is None:
                print(f"   {method:<20} UNAVAILABLE")
            else:
                print(f"   {method:<20} mean Δ {s['mean']:+.4f}  SD {s['sd']:.4f}  wins {s['wins']}/5  paired t {s['t']:+.2f} "
                      f"p {s['p']:.3f}  per-fold {s['diffs']}  -> {verdict_one(s)}")
        print(f"   {row['flag']}")


print("=" * 110 + "\nPRIMARY COMPARISONS (v30)\n" + "=" * 110)
compare("CA", "F0", "does real cross-attention beat plain concatenation, SAME new encoders (cleanest test)")
compare("CA", "D", "new small encoders vs old encoders, BOTH with D's cross-attention (isolates the encoders)")
compare("CAG", "CA", "does the gate after cross-attention help further?")
compare("CAG", "F1", "cross-attention-then-gate vs gate alone (same new encoders, same gate MLP)")
_win_tab = COMBINED[COMBINED.level == "window"].set_index("model").macro_f1
BEST = str(_win_tab.idxmax())
BEST_FUSION = str(_win_tab.drop("I").idxmax())
if BEST == "I":
    print(f"\nThe best model overall by pooled window macro-F1 is I itself ({_win_tab['I']:.4f}); comparing the best "
          f"fusion model ({BEST_FUSION}, {_win_tab[BEST_FUSION]:.4f}) against I.")
compare(BEST_FUSION if BEST == "I" else BEST, "I", f"best {'fusion ' if BEST == 'I' else ''}model vs IMU alone (the standing bar; "
        f"chosen by pooled macro-F1 on the test folds, so descriptive)")

print("\n" + "=" * 110 + "\nv29's COMPARISONS RE-CHECKED UNDER BOTH METHODS\n" + "=" * 110)
compare("F1", "F0", "v29: content gate vs plain concatenation")
compare("F0", "D", "v29: new encoders + no cross-attention vs old encoders + cross-attention (conflated)")
compare("F2", "F1", "v29: sigmoid vs softmax gate")
compare("F1", "I", "v29: best v29 fusion vs IMU")

_flags = [r for r in COMP_RESULTS if not r["flag"].startswith("methods agree")]
print(f"\n{len(_flags)} of {len(COMP_RESULTS)} comparison rows carry a flag:")
for r in _flags:
    print(f"   {r['comparison']} [{r['level']}]: {r['flag']}")

## 11. Cross-attention and gate diagnostics per fold (CA, CAG)
Each fold's test windows are summarized per seed:
- **Attention (a2i = audio queries attending to IMU; i2a = the reverse):** mean, SD, min and max of the attention
  entries; mean normalized row entropy (1.0 = uniform); mean max-weight ÷ uniform; and the across-window SD of the
  head-averaged map, relative to uniform.
- **Residual scales:** the learned γ_as and γ_sa, which scale how much cross-attended content is added.
- **CAG only:** the g_a / g_i distribution and the constant-gate effect.

In [ ]:
# ==============================================================================
# 11. DIAGNOSTICS PER FOLD
# ==============================================================================
_v30 = pd.read_csv(RESULTS_CSV)
for m in V30_MODELS:
    t = _v30[_v30.model == m].sort_values(["fold", "seed"])
    if t.empty:
        continue
    cols = ["fold", "fold_day", "seed", "gamma_as", "gamma_sa"] + [c for c in ATTN_COLS if c.split("_", 2)[2] in
                                                                   ("std", "max", "entropy_norm", "max_over_uniform", "across_window_sd_rel")]
    print(f"\n=== {m}: cross-attention on each fold's test windows ({ATTN_SAMPLE_WINDOWS} per fold; per seed) ===")
    print(t[cols].round(4).to_string(index=False))
    for d, lab in (("a2i", "audio->IMU"), ("i2a", "IMU->audio")):
        e = t[f"attn_{d}_entropy_norm"]
        print(f"   {lab}: row entropy {e.mean():.4f} of uniform (range {e.min():.4f}-{e.max():.4f}) -> "
              + ("NEAR-UNIFORM in every run" if (e > NEAR_UNIFORM_ENTROPY).all() else
                 "non-uniform (content-dependent) in " + f"{int((e <= NEAR_UNIFORM_ENTROPY).sum())}/{len(e)} runs"))
    print(f"   tau-invariant in {int(t.tau_invariant.astype(bool).sum())}/{len(t)} runs (no timestamp information used)")
    if m == "CAG":
        g = t[["fold", "seed", "g_a_mean", "g_a_std", "g_a_min", "g_a_max", "g_i_mean", "g_i_std", "gate_effect_mean_abs_dp",
               "gate_effect_argmax_changed"]].copy()
        g["varies (sd > thr)"] = np.maximum(t.g_a_std, t.g_i_std).values > GATE_STD_THRESHOLD
        print("\n=== CAG gate distribution on each fold's test windows ===")
        print(g.round(4).to_string(index=False))
hook("no-timestamps", bool(_v30.tau_invariant.astype(bool).all()) if len(_v30) else False,
     f"trained CA/CAG: test predictions bit-identical with every tau set to 0 in all {len(_v30)} logged runs", stop=False)
_masks = {m: [load_probs(m, k, s)["audio_mask"] for k in range(len(DAYS)) for s in SEEDS if load_probs(m, k, s) is not None] for m in V30_MODELS}
FEAT_NAMES = [f"ch{c + 1}/bin{b}" for c in range(len(KEEP_SLOTS)) for b in KEEP_BINS]
for m, ms in _masks.items():
    if ms:
        mm = np.mean(ms, axis=0)
        print(f"{m} learned audio mask, mean of {len(ms)} runs: top 10 " + ", ".join(f"{FEAT_NAMES[j]} {mm[j]:.3f}" for j in np.argsort(-mm)[:10]))

## 12. Summary, verdict and verification hooks
The verdict rule is v29's, fixed in advance: a difference counts as real only if mean Δ > 0.016 **and** wins ≥ 4/5.
Otherwise it is "no reliable difference". It is applied to **both** methods. When they disagree, both are stated and
no single headline is given.

In [ ]:
# ==============================================================================
# 12. SUMMARY (generated from the numbers) + HOOK TABLE
# ==============================================================================
def headline(comp, level="window"):
    r = next((x for x in COMP_RESULTS if x["comparison"] == comp and x["level"] == level), None)
    if r is None:
        return f"{comp}: not computed"
    parts = []
    for method in ("pooled-probability", "seed-averaged-F1"):
        s = r[method]
        parts.append(f"{method}: UNAVAILABLE" if s is None else
                     f"{method}: Δ {s['mean']:+.4f}, wins {s['wins']}/5, p {s['p']:.3f} -> {verdict_one(s)}")
    return f"{comp} [{level}] — " + " | ".join(parts) + f"   [{r['flag']}]"


_v30 = pd.read_csv(RESULTS_CSV)
L = ["WHAT v30 TELLS US", "",
     f"Protocol identical to v29: paired-window sha1 {PAIRED_SHA} (v29 {V29_PAIRED_SHA}), 95 missions, 5 day-out folds, "
     f"inner validation {INNER_VAL_RULE}, seeds {SEEDS}. v29's six models LOADED from {V29_CSV_PATH.name} / "
     f"{V29_TABLE_PATH.name} (not retrained). v30 runs logged: {len(_v30)}/{len(PLAN)}, training time "
     f"{_v30.train_seconds.sum() / 60 if len(_v30) else 0:.0f} min.", "",
     "Pooled day-out macro-F1 [95% mission-bootstrap CI] — window | mission:"]
for m in ALL_MODELS:
    w = COMBINED[(COMBINED.model == m) & (COMBINED.level == "window")]
    mi = COMBINED[(COMBINED.model == m) & (COMBINED.level == "mission")]
    if len(w):
        L.append(f"  {m:<4} {w.macro_f1.iloc[0]:.3f} [{w.f1_ci_low.iloc[0]:.3f}, {w.f1_ci_high.iloc[0]:.3f}] | "
                 f"{mi.macro_f1.iloc[0]:.3f} [{mi.f1_ci_low.iloc[0]:.3f}, {mi.f1_ci_high.iloc[0]:.3f}]   "
                 f"({(N_PARAMS[m] if m in V30_MODELS else int(_v29_params[m])):,} params){'  <- v30' if m in V30_MODELS else ''}")
L += ["", "Paired comparisons (5 folds; both methods):"]
for comp in ["CA − F0", "CA − D", "CAG − CA", "CAG − F1", f"{BEST_FUSION if BEST == 'I' else BEST} − I"]:
    L.append("  " + headline(comp))
L += ["", "v29 comparisons re-checked: " + "; ".join(headline(c).split(" — ")[0] + ": " + next(
    (x["flag"] for x in COMP_RESULTS if x["comparison"] == c and x["level"] == "window"), "") for c in ["F1 − F0", "F0 − D", "F2 − F1", "F1 − I"])]
for m in V30_MODELS:
    t = _v30[_v30.model == m]
    if len(t):
        L.append(f"{m} attention: row entropy a2i {t.attn_a2i_entropy_norm.mean():.3f}, i2a {t.attn_i2a_entropy_norm.mean():.3f} of "
                 f"uniform; gamma_as {t.gamma_as.mean():.3f}, gamma_sa {t.gamma_sa.mean():.3f}; tau-invariant "
                 f"{int(t.tau_invariant.astype(bool).sum())}/{len(t)}.")
L += ["", "Determinism: established in v29 (mamba_ssm CUDA backward jitter ~1e-11 in gradients, 0.00 effect on validation "
          "macro-F1); not re-run.",
      "Caveats: 5 folds x 2 seeds -> paired tests have df = 4 and are weak; BEST vs I picks BEST on the test folds (descriptive); "
      "CA − D isolates the encoder change (cross-attention on both sides), unlike v29's F0 − D."]
SUMMARY = "\n".join(L)
print(SUMMARY)
(WORK / "v30_summary.txt").write_text(SUMMARY, encoding="utf-8")

print("\n" + "=" * 100 + "\nVERIFICATION HOOKS\n" + "=" * 100)
for name, res in HOOKS.items():
    print(f"  {name:<14} {'PASS' if all(ok for ok, _ in res) else 'FAIL'}")
    for ok, msg in res:
        print(f"      {'PASS' if ok else 'FAIL'}: {msg}")